[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.en/cap08/cap08.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

## 💻 **Practical Part with Programming Exercises**

The present list of programming exercises (PE) consolidates the theoretical formulations presented throughout Chapter 8 — Feature Matching, Object Detection, and Classical Segmentation — through an applied practical track. As in the previous chapter, the PEs isolate the **intermediate quantities** of each technique — the distance between binary descriptors, the terms of an integral image, the count of *inliers* of a candidate model, the overlap between bounding boxes, and the label of each connected component — allowing each step of the reasoning to be manually validated without relying on OpenCV or external images.

The sequencing of the exercises reproduces the conceptual flow of the chapter: it begins with the **Hamming distance**, the core of binary descriptor matching such as ORB; it advances to the counting of ***inliers*** that underpins **RANSAC** in the robust estimation of a homography; it proceeds with the **integral image**, the computational trick that makes ***Haar Cascade*** viable in real time; it delves into **IoU and Non-Maximum Suppression**, the post-processing common to virtually every object detector; and it concludes with **connected component labeling**, the classical approach — and its limitations — for segmenting individual instances in a binary mask.

### 🎯 Objective of this Notebook

This notebook allows you to develop, validate, organize, and test solutions for **Programming Exercises (EPs)** in interactive environments, such as Colab, using the same test cases as Moodle, and copying them there only when registering the official grade.

### *Download*

Download `morph.py` and `testsuite.py` by running the cell below:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Running the Tests
To evaluate the tests, run `TestSuite("EP08_01.extension").run()` in a new cell, replacing the extension with that of the language used (`.py`, `.java`, `.c`, `.cpp`, `.js`, or `.r`). The system downloads the test cases from GitHub, runs the program, and calculates the grade automatically.

To test Python code directly, without saving a file, use `run_code(code)` passing the code as a *string* in a variable `code`:

```python
code = """
# ... your code here ...
"""
TestSuite("EP08_01").run_code(code)
```

### 🛠️ Summary of Methods in `morph.py` (Ch. 8)

The `morph.py` library provides functions for connected component analysis, contour extraction, geometric metrics, and annotations:

1. **Components and Contours (`connectedComponents`, `findContours`)**
Label connected regions and extract contours from binary images.
2. **Contour Properties (`contourArea`, `arcLength`, `convexHull`, `approxPolyDP`, `fitLine`)**
Compute area, perimeter, convex hull, polygonal approximation, and line fitting for a contour.
3. **Enclosing Geometry (`boundingRect`, `minAreaRect`, `boxPoints`, `minEnclosingCircle`, `fitEllipse`)**
Determine enclosing rectangles (aligned or oriented), ellipses, and the smallest bounding circle.
4. **Measurement Extraction and Persistence (`measure`, `saveMeasures`)**
Extract geometric descriptors of objects (area, circularity, solidity, centroid) and export the data to CSV, text, or YOLO format.
5. **Evaluation and Visualization (`IoU`, `verifyBoundBox`, `showBoundBox`)**
Compute the Intersection over Union, validate bounding boxes against ground truths, and draw annotated bounding boxes on the image.

### EP08_01 🟢 Hamming Distance and Binary Descriptor Matching

ORB, used in Practical Project 1 of this chapter, describes the neighborhood of each keypoint as a sequence of bits — and, therefore, the comparison between two descriptors does not use the Euclidean distance of the k-NN from Chapter 7, but rather the **Hamming distance**: the number of positions in which the bits differ. Before calling `cv2.BFMatcher(cv2.NORM_HAMMING)`, you were tasked with manually implementing this brute-force matching — the same step that, when executed internally by OpenCV, precedes the robust homography estimation by RANSAC.

#### 📋 Implementation Guidelines

1. **Quantities:** Read the integers $N$ and $M$ — the number of descriptors extracted from image A and image B, respectively.
2. **Descriptors from A:** Read $N$ lines, each containing a binary descriptor (a *string* of characters `0` and `1`, all of the same length).
3. **Descriptors from B:** Read $M$ lines, in the same format.
4. **Threshold:** Read the integer $\tau$ — the maximum acceptable Hamming distance for a match to be considered valid.
5. **Hamming Distance:** For two binary descriptors $a$ and $b$ of the same length,
$$
d_H(a, b) = \sum_{k} \mathbb{1}[a_k \neq b_k],
$$
   i.e., the count of positions in which the bits differ.
6. **Nearest-neighbor matching:** For each descriptor $a_i$ from A ($i$ in reading order, starting at $0$), compute its Hamming distance to **all** descriptors from B and find the one with the smallest distance. In case of a tie between two or more descriptors from B with the same minimum distance, choose the one with the **smallest index**.
7. **Threshold filtering:** If the smallest distance found is $\le \tau$, the match is valid; otherwise, $a_i$ has no corresponding match.
8. **Output:** For each $i$ from $0$ to $N-1$, in reading order, print a line: `i j d` if there is a valid match (where $j$ is the index of the chosen descriptor from B and $d$ its distance), or `i -1` otherwise. At the end, print `Total correspondências válidas: X`.

#### 📌 Computational Constraints

* **Same length:** all descriptors (from A and B) have exactly the same number of bits.
* **Brute force:** compare each descriptor from A to **all** those from B — no indexing or acceleration structure is required.
* **Tie-breaking by smallest index in B**, and **never** by the reading order of A (which is already natural, since each $a_i$ is handled independently).

#### 🧠 Theoretical Foundation

| Element | Role in ORB matching |
|---|---|
| Binary descriptor (BRIEF) | Each bit is the result of an intensity comparison between two pixels in the neighborhood |
| Hamming distance | Dissimilarity metric between binary *strings*; much faster to compute than the Euclidean distance (XOR operation + bit counting) |
| Nearest neighbor | Matching criterion: each point from A is paired with the point from B whose descriptor is most similar |
| Threshold $\tau$ | Filters unreliable matches even before RANSAC — but, as discussed in the chapter, some incorrect matches still pass, requiring the robustness of RANSAC |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $N$ and $M$.
* Next $N$ lines: one binary descriptor per line (*string* of `0`s and `1`s).
* Next $M$ lines: one binary descriptor per line, in the same format.
* Last line: Integer $\tau$.

**Output:**

* $N$ lines, one per descriptor from A, in the format `i j d` or `i -1`.
* Last line: `Total correspondências válidas: X`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 3 3<br>10101010<br>11110000<br>00001111<br>10101011<br>00001110<br>11111111<br>2 | 0 0 1<br>1 -1<br>2 1 1<br>Total correspondências válidas: 2 | The descriptor `11110000` has no match: its nearest neighbor is at distance 4, above the threshold $\tau=2$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0801" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0801 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0801 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0801 button:hover { background: #e8dfcf; }
  .sim-ep0801_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0801_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0801_bit { width: 36px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-family: monospace; font-weight: 700; font-size: 14px; user-select: none; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP08_01: Hamming Distance between Binary Descriptors</span>
  <span class="sim-ep0801_pill">8-Bit Descriptors</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel Informativo de Instruções -->
  <div class="sim-ep0801_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center;">
      Click any bit of the <b>Descriptor B</b> to flip it and watch the Hamming distance change in real time.
    </div>
  </div>

  <!-- Grades dos Descritores -->
  <div class="sim-ep0801_panel" style="margin-bottom:14px; display:flex; flex-direction:column; gap:12px; align-items:center;">
    <div>
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:6px; text-align:center; letter-spacing:0.04em;">
        Descriptor A (Fixed)
      </div>
      <div id="sim-ep0801_a" style="display:grid; grid-template-columns:repeat(8, 36px); gap:4px;"></div>
    </div>

    <div>
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:6px; text-align:center; letter-spacing:0.04em;">
        Descriptor B (Click to Flip)
      </div>
      <div id="sim-ep0801_b" style="display:grid; grid-template-columns:repeat(8, 36px); gap:4px;"></div>
    </div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0801_debug" class="sim-ep0801_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep01(root){
    if (!root || root.dataset.sim08Ep01Init) return;
    root.dataset.sim08Ep01Init = "1";

    var A = [1, 0, 1, 0, 1, 0, 1, 0];
    var B = [1, 0, 1, 0, 1, 0, 1, 1];

    var aEl = root.querySelector('#sim-ep0801_a');
    var bEl = root.querySelector('#sim-ep0801_b');
    var dbg = root.querySelector('#sim-ep0801_debug');

    function estiloBit(destacado, interativo){
      var base = destacado 
        ? 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;' 
        : 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
      var cursor = interativo ? ' cursor:pointer;' : ' cursor:default;';
      return base + cursor;
    }

    function render(){
      aEl.innerHTML = ''; 
      bEl.innerHTML = '';
      var dist = 0;

      for (var k = 0; k < 8; k++){
        var diff = A[k] !== B[k];
        if (diff) dist++;

        var da = document.createElement('div');
        da.className = 'sim-ep0801_bit';
        da.style.cssText = estiloBit(diff, false);
        da.textContent = A[k];
        aEl.appendChild(da);

        var db = document.createElement('div');
        db.className = 'sim-ep0801_bit';
        db.style.cssText = estiloBit(diff, true);
        db.textContent = B[k];
        
        (function(idx){
          db.addEventListener('click', function(){
            B[idx] = 1 - B[idx];
            render();
          });
        })(k);

        bEl.appendChild(db);
      }

      if (dist === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#26241d';
      }

      dbg.textContent = 'A = ' + A.join('') + '   B = ' + B.join('') + '   →   Distância de Hamming = ' + dist;
    }

    render();
  }

  function tryInitSim08Ep01(){
    var root = document.getElementById('sim-ep0801');
    if (root) initSim08Ep01(root); else setTimeout(tryInitSim08Ep01, 200);
  }
  tryInitSim08Ep01();
})();
</script>
""")

**Figure 8.1:** EP08_01 Simulator: Hamming Distance Between Two Binary Descriptors


<figure id="fig-08-sim-ep0801">
  <img src="imagens/fig-08-sim-ep0801.png" alt=" EP08_01 Simulator: Hamming Distance Between Two Binary Descriptors " style="max-width:80%" />
  <figcaption><strong>Figure 8.1:</strong>  EP08_01 Simulator: Hamming Distance Between Two Binary Descriptors </figcaption>
</figure>

In [ ]:
%%writefile EP08_01.py
# Python code

In [ ]:
TestSuite("EP08_01.py").run()

### EP08_02 🟢 Homography and RANSAC: Voting by *Inliers*

RANSAC, introduced in the section "Mathematical Modeling: Homography and RANSAC," repeats a cycle of three steps — sampling a minimal set, estimating a candidate model, and counting how many correspondences are consistent with it (the ***inliers***) — keeping at the end the most voted model. The model estimation step from 4 points (step 2) involves linear algebra that is beyond the scope of this assignment; here, you are directly given a set of **already candidate** homographies — as if each one had been estimated from a different random sample — and you are tasked with reproducing exactly the algorithm's decisive step: **apply each model to all correspondences and count its *inliers***, choosing the winner.

#### 📋 Implementation Guidelines

1. **Correspondences:** Read the integer $N$ and then $N$ lines with four real numbers each, $x\ y\ x'\ y'$ — a point from image A and its (possibly incorrect) corresponding point in image B, exactly as produced by the matching step of EP08_01.
2. **Candidate models:** Read the integer $K$ (number of candidate homographies) and the real number $\varepsilon$ (reprojection error threshold). Then, read $K$ lines, each with nine real numbers $h_{11}\ h_{12}\ h_{13}\ h_{21}\ h_{22}\ h_{23}\ h_{31}\ h_{32}\ h_{33}$ — the elements of the candidate matrix $H$, in row-major reading order.
3. **Reprojection:** For each correspondence $(x,y,x',y')$ and each candidate model $H_k$, compute the projected point
$$
\begin{bmatrix} \hat x \\ \hat y \\ \hat w \end{bmatrix} = H_k \begin{bmatrix} x \\ y \\ 1 \end{bmatrix},
\qquad
(\hat x / \hat w,\ \hat y / \hat w)\ \text{is the projected point.}
$$
4. **Reprojection error:** $e = \sqrt{(\hat x/\hat w - x')^2 + (\hat y /\hat w - y')^2}$.
5. **Inlier counting:** A correspondence is an *inlier* of model $H_k$ if $e \le \varepsilon$.
6. **Best model selection:** The winning model is the one with the largest number of *inliers*; in case of a tie, choose the one with the **smallest index** $k$ (the first one encountered during the RANSAC iterative cycle).
7. **Output:** For each model $k$ from $0$ to $K-1$, in reading order, print `Modelo k: I inliers`. Finally, print `Melhor modelo: k_best com I_best inliers`.

#### 📌 Computational Constraints

* **Inclusive comparison:** a reprojection error **exactly equal** to $\varepsilon$ counts as an *inlier* ($e \le \varepsilon$).
* **No estimation of $H$:** the matrices are already provided ready-made — there is no need (nor expectation) to solve any linear system.
* **Tie resolved by the smallest index**, reflecting the natural behavior of an iterative algorithm that traverses models in order and only replaces the best found so far when a new model **strictly surpasses** it.

#### 🧠 Theoretical Foundation

| Element | Role in RANSAC |
|---|---|
| Minimal sample (4 pairs) | Sufficient to determine the 8 degrees of freedom of a homography |
| Candidate model $H_k$ | Estimated from a minimal sample; can be good or bad, depending on whether the sample contained *outliers* |
| Reprojection error | Measures how well the model "predicts" each observed correspondence |
| *Inlier* vs. *outlier* | Correspondences consistent with the winning model (*inliers*) vs. the remaining ones, typically incorrect matches from the *matching* step |
| Final refinement | In practice, after selecting the best model, RANSAC recomputes it using **only** its *inliers* — a step not required in this assignment |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $N$.
* Next $N$ lines: four real numbers $x\ y\ x'\ y'$.
* Next line: Integer $K$ and real number $\varepsilon$.
* Next $K$ lines: nine real numbers (elements of $H_k$, row-major).

**Output:**

* $K$ lines in the format `Modelo k: I inliers`.
* Last line: `Melhor modelo: k_best com I_best inliers`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 5<br>0 0 0 0<br>1 1 2 2<br>2 0 4 0<br>0 2 0 4<br>5 5 1 1<br>2 0.5<br>2 0 0 0 2 0 0 0 1<br>1 0 0 0 1 0 0 0 1 | Modelo 0: 4 inliers<br>Modelo 1: 1 inliers<br>Melhor modelo: 0 com 4 inliers | Model 0 (scale ×2) correctly explains 4 of the 5 correspondences; the 5th, $(5,5)\to(1,1)$, is an *outlier* that neither model explains well. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0802" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0802 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0802 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0802 button:hover { background: #e8dfcf; }
  #sim-ep0802 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0802_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0802_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP08_02: RANSAC &mdash; Inlier Count</span>
  <span class="sim-ep0802_pill">Model: Scale &times;2</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0808_panel sim-ep0802_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Threshold (&epsilon;): <span id="sim-ep0802_vl" style="font-family:monospace; color:#26241d;">0.50</span>
      </label>
    </div>
    
    <input id="sim-ep0802_sl" type="range" min="0" max="13" step="0.25" value="0.5">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      The candidate model maps (x,y) &rarr; to (2x,2y). Adjust the threshold &epsilon; and see which correspondences become inliers or outliers.
    </div>
  </div>

  <!-- Cards de Pontos / Correspondências -->
  <div id="sim-ep0802_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:8px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0802_debug" class="sim-ep0802_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep02(root){
    if (!root || root.dataset.sim08Ep02Init) return;
    root.dataset.sim08Ep02Init = "1";

    var pontos = [
      {x:0, y:0, xp:0, yp:0},
      {x:1, y:1, xp:2, yp:2},
      {x:2, y:0, xp:4, yp:0},
      {x:0, y:2, xp:0, yp:4},
      {x:5, y:5, xp:1, yp:1}
    ];

    var slEl  = root.querySelector('#sim-ep0802_sl');
    var vlEl  = root.querySelector('#sim-ep0802_vl');
    var cards = root.querySelector('#sim-ep0802_cards');
    var dbg   = root.querySelector('#sim-ep0802_debug');

    function render(){
      var eps = parseFloat(slEl.value);
      vlEl.textContent = eps.toFixed(2);
      cards.innerHTML = '';
      var inliers = 0;

      pontos.forEach(function(p){
        var px = 2 * p.x, py = 2 * p.y;
        var erro = Math.sqrt((px - p.xp) * (px - p.xp) + (py - p.yp) * (py - p.yp));
        var dentro = erro <= eps;
        if (dentro) inliers++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
            : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">(' + p.x + ',' + p.y + ') &rarr; (' + p.xp + ',' + p.yp + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">erro = ' + erro.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'INLIER' : 'outlier') + '</div>';

        cards.appendChild(div);
      });

      if (inliers > 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = '\u03B5 = ' + eps.toFixed(2) + '  |  inliers = ' + inliers + ' de ' + pontos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim08Ep02(){
    var root = document.getElementById('sim-ep0802');
    if (root) initSim08Ep02(root); else setTimeout(tryInitSim08Ep02, 200);
  }
  tryInitSim08Ep02();
})();
</script>
""")

**Figure 8.2:** EP08_02 Simulator: RANSAC — Voting by Inliers among Candidate Models


<figure id="fig-08-sim-ep0802">
  <img src="imagens/fig-08-sim-ep0802.png" alt=" EP08_02 Simulator: RANSAC — Voting by Inliers among Candidate Models " style="max-width:80%" />
  <figcaption><strong>Figure 8.2:</strong>  EP08_02 Simulator: RANSAC — Voting by Inliers among Candidate Models </figcaption>
</figure>

In [ ]:
%%writefile EP08_02.py
# Python code

In [ ]:
TestSuite("EP08_02.py").run()

### EP08_03 🟢 Integral Image: Rectangular Sums in Constant Time

Imagine a security camera processing 30 frames per second, and for each frame the system must scan the image at dozens of positions and scales, testing at each one a set of rectangular features to decide "is there a face here?". If computing the sum of intensities for each rectangle required summing pixel by pixel, the system would have no chance of real-time evaluation — the bottleneck would lie precisely in the most repetitive part of the algorithm. It is exactly this bottleneck that the integral image eliminates.

The Haar Cascade evaluates thousands of rectangular features per window, at multiple positions and scales — something unfeasible in real time if each rectangle required summing its pixels one by one. The **integral image**, defined in the section on Haar Cascade, solves this problem: once precomputed, the sum of intensities of **any** rectangular region is obtained with just four lookups and three arithmetic operations, regardless of the size of the rectangle.

You have been tasked with implementing this structure from scratch: first, compute the integral image from the original image; then, answer arbitrary rectangular queries using it.

#### 📋 Implementation Guidelines

1. **Input:** Read the dimensions $H \times W$ of the image and its $H \times W$ integer intensity values.
2. **Integral image:** Compute, for each position $(i,j)$ (indexing starting from $0$, `[row][column]`),
$$
II(i,j) = \sum_{i' \le i,\ j' \le j} I(i', j'),
$$
   that is, the sum of all pixels above and to the left of $(i,j)$, including the position itself.
3. **Queries:** Read the integer $Q$ and then $Q$ lines, each with four integers $x_1\ y_1\ x_2\ y_2$ — the top-left and bottom-right corners of a rectangle, **both inclusive**, with $0 \le x_1 \le x_2 < W$ and $0 \le y_1 \le y_2 < H$.
4. **Rectangular sum in O(1):** For each query, compute the sum of intensities within the rectangle using exclusively values already present in $II$ (without traversing the original pixels):
$$
S(x_1,y_1,x_2,y_2) = II(y_2,x_2) - II(y_2, x_1{-}1) - II(y_1{-}1, x_2) + II(y_1{-}1, x_1{-}1),
$$
   treating any term with a row or column index equal to $-1$ as $0$.
5. **Output:** First, print the complete integral image — $H$ lines with $W$ integers each. Then, for each query, print a single integer: the sum of the corresponding region.

#### 📌 Computational Constraints

* **Do not recalculate by brute force:** the answer to each query must use the four-term formula on $II$, not a direct sum of the rectangle's pixels (even though the numerical result is the same, the point of the exercise is precisely this technique).
* **Rectangles with inclusive coordinates:** $(x_1,y_1)$ and $(x_2,y_2)$ belong to the summed region.
* **Boundary handling:** when querying $II$ with index $-1$ (when $x_1=0$ or $y_1=0$), use the value $0$.

#### 🧠 Theoretical Foundation

| Element | Role in Haar Cascade |
|---|---|
| Integral image $II$ | Precomputed once per image, in time $O(HW)$ |
| O(1) query | Each Haar feature (difference between sums of rectangular regions) is evaluated with few operations, regardless of the rectangle's area |
| Scalability | It is this constancy that makes it feasible to evaluate thousands of features, at multiple positions and scales, in real time |
| Inclusion-exclusion principle | The four terms of the formula sum the desired region and subtract exactly the areas counted in excess |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $H$ and $W$.
* Next $H$ lines: $W$ integers each (original image).
* Next line: Integer $Q$.
* Next $Q$ lines: four integers $x_1\ y_1\ x_2\ y_2$.

**Output:**

* $H$ lines with $W$ integers each (the integral image).
* $Q$ lines, one per query, with the sum of the corresponding region.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>1<br>0 0 2 2 | 1 3 6<br>5 12 21<br>12 27 45<br>45 | The query covers the entire image; the sum coincides with $II(2,2)$ and with the sum of all 9 values. |
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>2<br>1 1 2 2<br>0 0 1 1 | 1 3 6<br>5 12 21<br>12 27 45<br>28<br>12 | The first query uses the four terms of the formula; the second coincides directly with $II(1,1)$, since it starts at the origin. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0803" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0803 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0803 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0803 button:hover { background: #e8dfcf; }
  #sim-ep0803 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0803_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0803_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP08_03: Rectangular Sum with Integral Image</span>
  <span id="sim-ep0803_badge" class="sim-ep0803_pill">Internal</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Botões de Preset -->
  <div class="sim-ep0803_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Choose a rectangle (x<sub>1</sub>, y<sub>1</sub>) &ndash; (x<sub>2</sub>, y<sub>2</sub>). The integral image II includes virtual border (&minus;1) with zeros for exception-free validation.
    </div>

    <div style="display:flex; gap:6px; justify-content:center; flex-wrap:wrap;">
      <button data-preset="0,0,3,3">From Origin</button>
      <button data-preset="0,1,2,3">Left Edge</button>
      <button data-preset="1,0,3,2">Top Edge</button>
      <button data-preset="1,1,2,2">Fully Internal</button>
      <button data-preset="2,2,2,2">Single Pixel</button>
    </div>
  </div>

  <!-- Sliders de Seleção das Coordenadas -->
  <div class="sim-ep0803_panel" style="margin-bottom:14px;">
    <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:12px;">
      
      <div>
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:4px;">
          Upper-Left Corner (x<sub>1</sub>, y<sub>1</sub>) = <span id="sim-ep0803_v_tl" style="font-family:monospace; color:#26241d;">(1,1)</span>
        </div>
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600;">x<sub>1</sub></div>
        <input id="sim-ep0803_x1" type="range" min="0" max="3" step="1" value="1">
        
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600; margin-top:4px;">y<sub>1</sub></div>
        <input id="sim-ep0803_y1" type="range" min="0" max="3" step="1" value="1">
      </div>

      <div>
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:4px;">
          Lower-Right Corner (x<sub>2</sub>, y<sub>2</sub>) = <span id="sim-ep0803_v_br" style="font-family:monospace; color:#26241d;">(2,2)</span>
        </div>
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600;">x<sub>2</sub></div>
        <input id="sim-ep0803_x2" type="range" min="0" max="3" step="1" value="2">
        
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600; margin-top:4px;">y<sub>2</sub></div>
        <input id="sim-ep0803_y2" type="range" min="0" max="3" step="1" value="2">
      </div>

    </div>
  </div>

  <!-- Grades das Matrizes -->
  <div style="display:flex; gap:20px; justify-content:center; flex-wrap:wrap; margin-bottom:14px;">
    <div class="sim-ep0803_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:8px; letter-spacing:0.04em;">
        Original Image I (4&times;4)
      </div>
      <div id="sim-ep0803_gridI" style="display:grid; justify-content:center;"></div>
    </div>

    <div class="sim-ep0803_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:8px; letter-spacing:0.04em;">
        Integral Image II (With Virtual Border &minus;1)
      </div>
      <div id="sim-ep0803_gridII" style="display:grid; justify-content:center;"></div>
    </div>
  </div>

  <!-- Legenda das Operações -->
  <div style="display:flex; gap:12px; justify-content:center; flex-wrap:wrap; margin-bottom:14px; font-size:10px; font-weight:700; color:#5e5a4a;">
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#2980b9; border-radius:2px; display:inline-block;"></span> + II(y<sub>2</sub>, x<sub>2</sub>)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#d35400; border-radius:2px; display:inline-block;"></span> &minus; II(y<sub>2</sub>, x<sub>1</sub>&minus;1)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#d35400; border-radius:2px; display:inline-block;"></span> &minus; II(y<sub>1</sub>&minus;1, x<sub>2</sub>)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#27ae60; border-radius:2px; display:inline-block;"></span> + II(y<sub>1</sub>&minus;1, x<sub>1</sub>&minus;1)</span>
  </div>

  <!-- Painéis Informativos / Resultados -->
  <div id="sim-ep0803_formula" class="sim-ep0803_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; margin-bottom:8px;"></div>
  <div id="sim-ep0803_verify" class="sim-ep0803_panel" style="font-family:monospace; font-size:11px; color:#04342C; background:#eafaf1; border-color:#a3e4d7; text-align:center;"></div>

</div>
</div>

<script>
(function(){
  function initSim08Ep03(root){
    if (!root || root.dataset.sim08Ep03Init) return;
    root.dataset.sim08Ep03Init = "1";

    var I = [
      [2, 1, 3, 4],
      [5, 6, 1, 2],
      [3, 2, 4, 1],
      [1, 3, 2, 5]
    ];
    var N = 4;
    var II = [];
    for (var i = 0; i < N; i++){ II.push([0, 0, 0, 0]); }
    
    for (var i = 0; i < N; i++){
      for (var j = 0; j < N; j++){
        II[i][j] = I[i][j] +
          (i > 0 ? II[i - 1][j] : 0) + 
          (j > 0 ? II[i][j - 1] : 0) - 
          (i > 0 && j > 0 ? II[i - 1][j - 1] : 0);
      }
    }

    var x1El     = root.querySelector('#sim-ep0803_x1');
    var y1El     = root.querySelector('#sim-ep0803_y1');
    var x2El     = root.querySelector('#sim-ep0803_x2');
    var y2El     = root.querySelector('#sim-ep0803_y2');
    var vTl      = root.querySelector('#sim-ep0803_v_tl');
    var vBr      = root.querySelector('#sim-ep0803_v_br');
    var gridI    = root.querySelector('#sim-ep0803_gridI');
    var gridII   = root.querySelector('#sim-ep0803_gridII');
    var formulaEl= root.querySelector('#sim-ep0803_formula');
    var verifyEl = root.querySelector('#sim-ep0803_verify');
    var badge    = root.querySelector('#sim-ep0803_badge');

    var CELL = 34, HEAD = 20;

    function cellDiv(text, size, extraStyle){
      var d = document.createElement('div');
      d.style.cssText = 'display:flex; align-items:center; justify-content:center; font-family:monospace; font-size:' + size + 'px;' + extraStyle;
      d.textContent = text;
      return d;
    }

    function clampAndRender(changed){
      var x1 = +x1El.value, y1 = +y1El.value, x2 = +x2El.value, y2 = +y2El.value;
      if (changed === 'x1' && x1 > x2) x2El.value = x1;
      if (changed === 'x2' && x2 < x1) x1El.value = x2;
      if (changed === 'y1' && y1 > y2) y2El.value = y1;
      if (changed === 'y2' && y2 < y1) y1El.value = y2;
      render();
    }

    function render(){
      var x1 = +x1El.value, y1 = +y1El.value, x2 = +x2El.value, y2 = +y2El.value;
      vTl.textContent = '(' + x1 + ',' + y1 + ')';
      vBr.textContent = '(' + x2 + ',' + y2 + ')';

      var sit;
      if (x1 === x2 && y1 === y2) sit = 'Pixel Único';
      else if (x1 === 0 && y1 === 0) sit = 'Desde a Origem';
      else if (x1 === 0) sit = 'Borda Esquerda';
      else if (y1 === 0) sit = 'Borda Superior';
      else sit = 'Interno';

      badge.textContent = sit;

      // Grade I
      gridI.style.gridTemplateColumns = HEAD + 'px repeat(' + N + ',' + CELL + 'px)';
      gridI.style.gridTemplateRows = HEAD + 'px repeat(' + N + ',' + CELL + 'px)';
      gridI.innerHTML = '';
      gridI.appendChild(cellDiv('', 10, 'color:#8a8371;'));
      for (var c = 0; c < N; c++) gridI.appendChild(cellDiv(c, 10, 'color:#8a8371; font-weight:700;'));
      
      for (var r = 0; r < N; r++){
        gridI.appendChild(cellDiv(r, 10, 'color:#8a8371; font-weight:700;'));
        for (var c = 0; c < N; c++){
          var dentro = (r >= y1 && r <= y2 && c >= x1 && c <= x2);
          gridI.appendChild(cellDiv(I[r][c], 12, 'border-radius:4px; transition:all 0.15s ease;' +
            (dentro 
              ? 'background:#f1ead7; border:2px solid #26241d; font-weight:700; color:#26241d;'
              : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;')));
        }
      }

      // Grade II (5x5 dados)
      var M = N + 1;
      gridII.style.gridTemplateColumns = HEAD + 'px repeat(' + M + ',' + CELL + 'px)';
      gridII.style.gridTemplateRows = HEAD + 'px repeat(' + M + ',' + CELL + 'px)';
      gridII.innerHTML = '';
      gridII.appendChild(cellDiv('', 10, 'color:#8a8371;'));
      for (var c2 = 0; c2 < M; c2++) gridII.appendChild(cellDiv(c2 - 1, 10, 'color:#8a8371; font-weight:700;'));

      var t1 = [y2 + 1, x2 + 1];
      var t2 = [y2 + 1, x1];
      var t3 = [y1, x2 + 1];
      var t4 = [y1, x1];

      function styleFor(r2, c2){
        var isVirtual = (r2 === 0 || c2 === 0);
        var base = isVirtual
          ? 'border-radius:4px; background:#fafaf7; border:1px dashed #e4dcc8; color:#8a8371;'
          : 'border-radius:4px; background:#fafaf7; border:1px solid #e4dcc8; color:#26241d;';
        
        function match(t, color, tcolor){
          if (r2 === t[0] && c2 === t[1]) {
            return 'border-radius:4px; font-weight:700; background:' + color + '; border:2px solid ' + tcolor + '; color:#ffffff;';
          }
          return null;
        }

        return match(t1, '#2980b9', '#1c5d85') || 
               match(t2, '#d35400', '#a04000') ||
               match(t3, '#d35400', '#a04000') || 
               match(t4, '#27ae60', '#1e8449') || base;
      }

      for (var r2 = 0; r2 < M; r2++){
        gridII.appendChild(cellDiv(r2 - 1, 10, 'color:#8a8371; font-weight:700;'));
        for (var c2 = 0; c2 < M; c2++){
          var val = (r2 === 0 || c2 === 0) ? 0 : II[r2 - 1][c2 - 1];
          gridII.appendChild(cellDiv(val, 12, styleFor(r2, c2)));
        }
      }

      function term(y, x){ return (y < 0 || x < 0) ? 0 : II[y][x]; }
      var a = term(y2, x2), b = term(y2, x1 - 1), c3 = term(y1 - 1, x2), d = term(y1 - 1, x1 - 1);
      var S = a - b - c3 + d;

      formulaEl.innerHTML =
        'S = II(' + y2 + ',' + x2 + ') &minus; II(' + y2 + ',' + (x1 - 1) + ') &minus; II(' + (y1 - 1) + ',' + x2 + ') + II(' + (y1 - 1) + ',' + (x1 - 1) + ')<br>' +
        'S = ' + a + ' &minus; ' + b + ' &minus; ' + c3 + ' + ' + d + ' = <b>' + S + '</b>';

      var direta = 0;
      for (var rr = y1; rr <= y2; rr++){
        for (var cc = x1; cc <= x2; cc++){
          direta += I[rr][cc];
        }
      }

      if (direta === S) {
        verifyEl.style.borderColor = '#a3e4d7';
        verifyEl.style.background  = '#eafaf1';
        verifyEl.style.color       = '#04342C';
      } else {
        verifyEl.style.borderColor = '#f5b7b1';
        verifyEl.style.background  = '#fdecea';
        verifyEl.style.color       = '#c0392b';
      }

      verifyEl.innerHTML = '&#10004; Verificação (Soma Direta dos Pixels) = ' + direta + (direta === S ? ' &rarr; Bate com S' : ' &rarr; Erro');
    }

    x1El.addEventListener('input', function(){ clampAndRender('x1'); });
    y1El.addEventListener('input', function(){ clampAndRender('y1'); });
    x2El.addEventListener('input', function(){ clampAndRender('x2'); });
    y2El.addEventListener('input', function(){ clampAndRender('y2'); });

    root.querySelectorAll('button[data-preset]').forEach(function(btn){
      btn.addEventListener('click', function(){
        var p = btn.getAttribute('data-preset').split(',').map(Number);
        x1El.value = p[0]; y1El.value = p[1]; x2El.value = p[2]; y2El.value = p[3];
        render();
      });
    });

    render();
  }

  function tryInitSim08Ep03(){
    var root = document.getElementById('sim-ep0803');
    if (root) initSim08Ep03(root); else setTimeout(tryInitSim08Ep03, 200);
  }
  tryInitSim08Ep03();
})();
</script>
""")

**Figure 8.3:** EP08_03 Simulator: Rectangular Sum in O(1) — Multiple Edge Cases


<figure id="fig-08-sim-ep0803">
  <img src="imagens/fig-08-sim-ep0803.png" alt=" EP08_03 Simulator: Rectangular Sum in O(1) — Multiple Edge Cases " style="max-width:80%" />
  <figcaption><strong>Figure 8.3:</strong>  EP08_03 Simulator: Rectangular Sum in O(1) — Multiple Edge Cases </figcaption>
</figure>

In [ ]:
%%writefile EP08_03.py
# Python code

In [ ]:
TestSuite("EP08_03.py").run()

### EP08_04 🟢 IoU and Non-Maximum Suppression (NMS)

The figure in this section showed the effect of Non-Maximum Suppression on a set of boxes produced by a *sliding window* detector: multiple redundant detections per object were reduced to a single box per object. You were tasked with reimplementing, byte by byte, the two functions that produced that result — `calcular_iou` and `supressao_nao_maximos` — to confirm, with your own hands, exactly the numbers presented in the chapter.

#### 📋 Implementation Guidelines

1. **Input:** Read the integer $N$ (number of boxes) and the real $\tau$ (IoU threshold). Then read $N$ lines, each with five reals $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.
2. **Intersection over Union:** For two boxes $A$ and $B$,
   $$
   \mathrm{IoU}(A,B) = \frac{\text{area}(A \cap B)}{\text{area}(A \cup B)},
   $$
   with the intersection area being zero when the boxes do not overlap.
3. **NMS Algorithm** (exactly as described in the chapter):
   
   a. Sort the boxes by `score` in descending order (ties preserve the original reading order).

   b. Select the highest-scoring box among the remaining ones; add it to the output and remove it from the list.

   c. Discard from the remaining list **all** boxes whose IoU with the selected box is **greater than or equal** to $\tau$ — only boxes with $\mathrm{IoU} < \tau$ remain as candidates.
   
   d. Repeat steps (b)–(c) until the list of remaining boxes is empty.

4. **Output:** For each retained box, in the order it was selected, print its original index (reading position, starting from $0$) and its `score`, with 2 decimal places. At the end, print `Total mantidas: X`.

#### 📌 Computational Constraints

* **Attention to the direction of the threshold:** contrary to what one might assume, a box is **suppressed** when $\mathrm{IoU} \ge \tau$ (not only when $\mathrm{IoU} > \tau$) — follow exactly this criterion, the same as the chapter's reference code.
* **Original indices:** the output references the reading position of each box in the input, not its position after sorting by `score`.
* **Area without the +1 pixel adjustment:** use area $= (x_{max}-x_{min}) \times (y_{max}-y_{min})$, exactly as in the chapter (without the "+1" adjustment sometimes used in other conventions).

#### 🧠 Theoretical Foundation

| Element | Role in Post-Processing |
|---|---|
| IoU | Quantifies the spatial overlap between two bounding boxes |
| *Sliding window* (Haar Cascade) | Typically produces multiple overlapping detections for the same object, at nearby positions and scales |
| Threshold $\tau$ | Controls the aggressiveness of suppression: too low merges nearby objects; too high lets redundancies pass |
| Sorting by `score` | Ensures that, among redundant boxes, the one with the highest confidence always survives |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $N$ and real $\tau$.
* Next $N$ lines: five reals $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.

**Output:**

* One line per retained box, in selection order: `index score` (score with 2 decimal places).
* Last line: `Total mantidas: X`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 5 0.4<br>50 50 150 150 0.90<br>60 55 155 145 0.75<br>58 60 160 150 0.60<br>300 300 400 420 0.95<br>310 305 395 415 0.70 | 3 0.95<br>0 0.90<br>Total mantidas: 2 | Exactly the example from the chapter's figure: 5 redundant boxes (2 objects) become 2 final detections. The IoU between the 1st and 2nd boxes is $\approx 0.775$, well above $\tau=0.4$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0804" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0804 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0804 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0804 button:hover { background: #e8dfcf; }
  #sim-ep0804 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0804_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0804_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP08_04: IoU and Non-Maximum Suppression (NMS)</span>
  <span class="sim-ep0804_pill">Suppress if IoU &ge; &tau;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0804_panel" style="margin-bottom:14px;">
    <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:12px;">
      
      <div>
        <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Candidate Offset (dx)</label>
          <span id="sim-ep0804_dx_v" style="font-family:monospace; font-weight:700; color:#26241d;">3</span>
        </div>
        <input id="sim-ep0804_dx" type="range" min="0" max="10" step="1" value="3">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Threshold (&tau;)</label>
          <span id="sim-ep0804_tau_v" style="font-family:monospace; font-weight:700; color:#26241d;">0.40</span>
        </div>
        <input id="sim-ep0804_tau" type="range" min="0.1" max="0.9" step="0.05" value="0.4">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      The blue box (higher score) has already been selected. Adjust the overlap and threshold &tau; to verify suppression of the red box (candidate).
    </div>
  </div>

  <!-- Canvas Visual de Caixas Delimitadoras -->
  <div class="sim-ep0804_panel" style="position:relative; width:100%; height:160px; margin-bottom:14px; overflow:hidden;">
    <div id="sim-ep0804_boxA" style="position:absolute; border:2px solid #2980b9; background:rgba(41,128,185,0.20); border-radius:4px; transition:all 0.15s ease;"></div>
    <div id="sim-ep0804_boxB" style="position:absolute; border:2px solid #c0392b; background:rgba(192,57,43,0.20); border-radius:4px; transition:all 0.15s ease;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0804_debug" class="sim-ep0804_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep04(root){
    if (!root || root.dataset.sim08Ep04Init) return;
    root.dataset.sim08Ep04Init = "1";

    var dxEl   = root.querySelector('#sim-ep0804_dx');
    var dxvEl  = root.querySelector('#sim-ep0804_dx_v');
    var tauEl  = root.querySelector('#sim-ep0804_tau');
    var tauvEl = root.querySelector('#sim-ep0804_tau_v');
    var boxA   = root.querySelector('#sim-ep0804_boxA');
    var boxB   = root.querySelector('#sim-ep0804_boxB');
    var dbg    = root.querySelector('#sim-ep0804_debug');

    var ESCALA = 10;
    var A = {x1: 5, y1: 3, x2: 15, y2: 13};

    function iou(a, b){
      var ix1 = Math.max(a.x1, b.x1), iy1 = Math.max(a.y1, b.y1);
      var ix2 = Math.min(a.x2, b.x2), iy2 = Math.min(a.y2, b.y2);
      var iw  = Math.max(0, ix2 - ix1), ih = Math.max(0, iy2 - iy1);
      var inter = iw * ih;
      var areaA = (a.x2 - a.x1) * (a.y2 - a.y1);
      var areaB = (b.x2 - b.x1) * (b.y2 - b.y1);
      return inter / (areaA + areaB - inter);
    }

    function render(){
      var dx  = parseInt(dxEl.value, 10);
      var tau = parseFloat(tauEl.value);

      dxvEl.textContent  = dx;
      tauvEl.textContent = tau.toFixed(2);

      var B = {x1: 5 + dx, y1: 3 + dx * 0.4, x2: 15 + dx, y2: 13 + dx * 0.4};

      boxA.style.left   = (A.x1 * ESCALA) + 'px';
      boxA.style.top    = (A.y1 * ESCALA) + 'px';
      boxA.style.width  = ((A.x2 - A.x1) * ESCALA) + 'px';
      boxA.style.height = ((A.y2 - A.y1) * ESCALA) + 'px';

      boxB.style.left   = (B.x1 * ESCALA) + 'px';
      boxB.style.top    = (B.y1 * ESCALA) + 'px';
      boxB.style.width  = ((B.x2 - B.x1) * ESCALA) + 'px';
      boxB.style.height = ((B.y2 - B.y1) * ESCALA) + 'px';

      var val = iou(A, B);
      var suprimida = val >= tau;

      if (suprimida) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'IoU(A,B) = ' + val.toFixed(4) + '  |  \u03C4 = ' + tau.toFixed(2) + '  \u2192  Candidata (Vermelha) ' + 
        (suprimida ? 'SUPRIMIDA (IoU \u2265 \u03C4)' : 'MANTIDA (IoU < \u03C4)');
    }

    dxEl.addEventListener('input', render);
    tauEl.addEventListener('input', render);
    render();
  }

  function tryInitSim08Ep04(){
    var root = document.getElementById('sim-ep0804');
    if (root) initSim08Ep04(root); else setTimeout(tryInitSim08Ep04, 200);
  }
  tryInitSim08Ep04();
})();
</script>
""")

**Figure 8.4:** EP08_04 Simulator: IoU and Non-Maximum Suppression


<figure id="fig-08-sim-ep0804">
  <img src="imagens/fig-08-sim-ep0804.png" alt=" EP08_04 Simulator: IoU and Non-Maximum Suppression " style="max-width:80%" />
  <figcaption><strong>Figure 8.4:</strong>  EP08_04 Simulator: IoU and Non-Maximum Suppression </figcaption>
</figure>

In [ ]:
%%writefile EP08_04.py
# Python code

In [ ]:
TestSuite("EP08_04.py").run()

### EP08_05 🟡 Connected Component Labeling: Instance Segmentation

The classic segmentation example in this chapter separated coin "instances" simply by their spatial disconnection in the binary mask resulting from Otsu's thresholding. That final step—labeling each connected component with an instance identifier—is exactly what you have been tasked with implementing here, from scratch, on an already prepared binary mask (0 = background, 1 = object), as if it were a manual reimplementation of `cv2.connectedComponents`.

This exercise also exposes, in a very concrete way, the limitation discussed in the chapter: the result depends entirely on how "neighborhood" between pixels is defined—and, as you will see in the second example, two diagonal pixels can be considered the same instance or different instances, depending solely on the chosen **connectivity**, not on any semantic notion of an object.

#### 📋 Implementation Guidelines

1. **Input:** Read the dimensions $H \times W$ of the binary mask and its $H \times W$ values ($0$ or $1$).
2. **Connectivity:** Read the integer $c \in \{4, 8\}$. Under 4-connectivity, the neighbors of $(i,j)$ are $(i{-}1,j)$, $(i{+}1,j)$, $(i,j{-}1)$, and $(i,j{+}1)$. Under 8-connectivity, the four diagonals are added: $(i{-}1,j{-}1)$, $(i{-}1,j{+}1)$, $(i{+}1,j{-}1)$, and $(i{+}1,j{+}1)$.
3. **Component discovery:** Traversing the mask in a row-by-row sweep, from left to right and top to bottom, whenever a pixel with value $1$ that is still unlabeled is found, it starts a **new component**: assign to it the next available label (the first discovered component receives label $1$, the second label $2$, and so on) and propagate this same label to all pixels with value $1$ reachable from it through a chain of neighbors (according to the chosen connectivity)—by breadth-first search, depth-first search, or *union-find*, at your discretion.
4. **Background pixels:** remain with label $0$ and do not belong to any instance.
5. **Output:** First, print the complete label map—$H$ lines with $W$ integers each. Then, for each label $\ell$ from $1$ to $K$ (in discovery order), print `Instance l: A pixels`, where $A$ is the number of pixels with that label. Finally, print `Total instances: K`.

#### 📌 Computational Constraints

* **Discovery order = sweep order:** labels are numbered in the order in which each new component is found by the row-wise sweep, not by size or position.
* **Explicit connectivity:** two pixels with value $1$ belong to the same instance only if there exists a chain of neighbors **according to $c$** linking one to the other—do not mistakenly use the opposite connectivity.
* **Pure binary mask:** all input values are exactly $0$ or $1$.

#### 🧠 Theoretical Background

| Element | Role in classic instance segmentation |
|---|---|
| Thresholding (Otsu, Chap. 4) | Previous stage that produces the binary mask from the intensity image |
| Connected component | Each instance is defined **solely** by the spatial connectivity of object pixels, without any notion of shape, class, or appearance |
| 4- vs. 8-connectivity | Parameter that alters the result: under 8-connectivity, two blobs joined only diagonally become a single instance |
| Central limitation | The technique merges instances that touch or overlap (even if they are clearly distinct objects), because there is no notion of "object"—only of "connected region" |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $H$ and $W$.
* Next $H$ lines: $W$ integers ($0$ or $1$) each.
* Last line: Integer $c$ ($4$ or $8$).

**Output:**

* $H$ lines with $W$ integers each (the label map).
* One line per instance, in discovery order: `Instance l: A pixels`.
* Last line: `Total instances: K`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | 0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 2 2<br>0 0 0 0 2 2<br>Instance 1: 4 pixels<br>Instance 2: 4 pixels<br>Total instances: 2 | Two clearly separated $2\times2$ blocks: the result is the same under 4- or 8-connectivity. |
| 2 2<br>1 0<br>0 1<br>8 | 1 0<br>0 1<br>Instance 1: 2 pixels<br>Total instances: 1 | Under 8-connectivity, the two diagonal pixels belong to the **same** instance. Repeat this example with $c=4$: the result changes to 2 instances of 1 pixel each—purely due to the change in connectivity, with no difference in the mask. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0805" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0805 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0805 button { font-size: 11px; padding: 6px 16px; border-radius: 20px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0805 button:hover { background: #e8dfcf; }
  #sim-ep0805 button.sim-ep0805_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0805_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0805_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP08_05: Connected Components (4 vs. 8 Connectivity)</span>
  <span class="sim-ep0805_pill">Same Mask &rarr; Different Labels</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles de Seleção de Conectividade -->
  <div class="sim-ep0805_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      The same mask (two diagonal pixels) &mdash; toggle the connectivity and observe the number of instances and the colors of the labels change.
    </div>

    <div style="display:flex; gap:8px; justify-content:center;">
      <button id="sim-ep0805_c4">4-Connectivity</button>
      <button id="sim-ep0805_c8" class="sim-ep0805_active">8-Connectivity</button>
    </div>
  </div>

  <!-- Exibição da Grade 2x2 -->
  <div class="sim-ep0805_panel" style="margin-bottom:14px; display:flex; justify-content:center;">
    <div id="sim-ep0805_grid" style="display:grid; grid-template-columns:repeat(2, 56px); gap:6px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0805_debug" class="sim-ep0805_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep05(root){
    if (!root || root.dataset.sim08Ep05Init) return;
    root.dataset.sim08Ep05Init = "1";

    var mask = [[1, 0], [0, 1]];
    var conect = 8;
    var CORES = ['#eafaf1', '#fdecea'];
    var BORDAS = ['#a3e4d7', '#f5b7b1'];
    var TEXTOS = ['#04342C', '#c0392b'];

    var btn4   = root.querySelector('#sim-ep0805_c4');
    var btn8   = root.querySelector('#sim-ep0805_c8');
    var gridEl = root.querySelector('#sim-ep0805_grid');
    var dbg    = root.querySelector('#sim-ep0805_debug');

    function rotula(){
      var H = mask.length, W = mask[0].length;
      var labels = [[0, 0], [0, 0]];
      var atual = 0;
      var viz4 = [[-1, 0], [1, 0], [0, -1], [0, 1]];
      var viz8 = viz4.concat([[-1, -1], [-1, 1], [1, -1], [1, 1]]);
      var viz = conect === 8 ? viz8 : viz4;

      for (var i = 0; i < H; i++){
        for (var j = 0; j < W; j++){
          if (mask[i][j] === 1 && labels[i][j] === 0){
            atual++;
            var fila = [[i, j]];
            labels[i][j] = atual;
            while (fila.length){
              var pos = fila.pop();
              var r = pos[0], c = pos[1];
              for (var k = 0; k < viz.length; k++){
                var nr = r + viz[k][0], nc = c + viz[k][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W && mask[nr][nc] === 1 && labels[nr][nc] === 0){
                  labels[nr][nc] = atual;
                  fila.push([nr, nc]);
                }
              }
            }
          }
        }
      }
      return {labels: labels, k: atual};
    }

    function estiloBotoes(){
      btn4.classList.toggle('sim-ep0805_active', conect === 4);
      btn8.classList.toggle('sim-ep0805_active', conect === 8);
    }

    function render(){
      var res = rotula();
      gridEl.innerHTML = '';

      for (var i = 0; i < 2; i++){
        for (var j = 0; j < 2; j++){
          var d = document.createElement('div');
          var lab = res.labels[i][j];
          var estilo = 'width:56px; height:56px; display:flex; align-items:center; justify-content:center; border-radius:8px; font-family:monospace; font-weight:700; font-size:13px; transition:all 0.15s ease;';
          
          if (lab === 0){
            estilo += 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;';
          } else {
            var idx = (lab - 1) % 2;
            estilo += 'background:' + CORES[idx] + '; border:2px solid ' + BORDAS[idx] + '; color:' + TEXTOS[idx] + ';';
          }

          d.style.cssText = estilo;
          d.textContent = mask[i][j] + (lab ? ' (r' + lab + ')' : '');
          gridEl.appendChild(d);
        }
      }

      estiloBotoes();

      if (res.k === 1) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#26241d';
      }

      dbg.textContent = 'Connectivity = ' + conect + '  \u2192  ' + res.k + ' Instância(s) Encontrada(s)';
    }

    btn4.addEventListener('click', function(){ conect = 4; render(); });
    btn8.addEventListener('click', function(){ conect = 8; render(); });

    render();
  }

  function tryInitSim08Ep05(){
    var root = document.getElementById('sim-ep0805');
    if (root) initSim08Ep05(root); else setTimeout(tryInitSim08Ep05, 200);
  }
  tryInitSim08Ep05();
})();
</script>
""")

**Figure 8.5:** Simulator EP08_05: Connected Components Labeling — 4 vs. 8 Connectivity


<figure id="fig-08-sim-ep0805">
  <img src="imagens/fig-08-sim-ep0805.png" alt=" Simulator EP08_05: Connected Components Labeling — 4 vs. 8 Connectivity " style="max-width:80%" />
  <figcaption><strong>Figure 8.5:</strong>  Simulator EP08_05: Connected Components Labeling — 4 vs. 8 Connectivity </figcaption>
</figure>

In [ ]:
%%writefile EP08_05.py
# Python code

In [ ]:
TestSuite("EP08_05.py").run()

### EP08_06 🟡 *Bounding Boxes*, Centroids, and Instance Properties with `mm.measure`

In the previous exercise (**EP08_05**), you can observe how segmentation by connected components labels contiguous binary regions to separate instances. However, for detection, tracking, and quantitative object analysis tasks, the simple label map is not sufficient. It becomes necessary to extract **spatial and geometric metrics** that characterize each instance individually.

This EP focuses on calculating and automatically extracting the fundamental computer vision properties for each connected component found in the binary mask, using the native `mm.measure(img)` method from the `morph` library:

1. **Bounding Box:** The smallest axis-aligned rectangle that completely encloses the instance, defined by its top-left corner $(x, y)$, width $w$, and height $h$.
2. **Geometric Centroid $(\bar{x}, \bar{y})$:** The center of mass of the instance on the discrete grid, equivalent to the first-order spatial moments $M_{10}/M_{00}$ and $M_{01}/M_{00}$.
3. **Geometric Contour Area ($A$):** The area enclosed by the instance contour, calculated via `mm.contourArea(c)`.



#### 📋 Implementation Guidelines

1. **Input:** Read the dimensions $H \times W$ of the binary mask, the $H \times W$ values ($0$ or $1$), and the connectivity parameter $c \in \{4, 8\}$.
2. **Automatic Extraction with `mm.measure`:** Pass the binarized image to the `mm.measure(img_bin)` function, which extracts OpenCV contours and returns a list of dictionaries containing the geometric properties of each instance.
3. **Returned Properties:** For each dictionary $m$ in the list returned by `medidas = mm.measure(img_bin)`:
   * **Area (`area`):** Numerical value of the geometric contour area `mm.contourArea(c)`.
   * **Bounding Box (`bbox`):** Tuple $(x, y, w, h)$ representing the top-left corner, width, and height.
   * **Centroid (`center`):** Tuple $(c_x, c_y)$ with the coordinates of the center of mass $M_{10}/M_{00}$ and $M_{01}/M_{00}$. Format with **two decimal places**.
4. **Output:** For each instance $1, \dots, K$ found (ordered by discovery/position in the image), print a line containing its properties. Finally, print the total number of instances.
   - For sorting, use `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`.



#### 🧠 Theoretical Foundation

| Property in `mm.measure` | Mathematical Calculation / Discrete Logic | Practical Application in Vision |
| --- | --- | --- |
| **`bbox` (OpenCV)** | $[x, y, w, h] = [\min(c), \min(r), \Delta c + 1, \Delta r + 1]$ | Classic OpenCV format. *Note: networks such as YOLO convert this rectangle to $(c_x, c_y, w, h)$ normalized.* |
| **`center`** | $\bar{x} = \frac{M_{10}}{M_{00}}, \quad \bar{y} = \frac{M_{01}}{M_{00}}$ | Exact center of mass of the mask (used in tracking and trajectory analysis). |
| **`area`** | $A = \text{contourArea}(C)$ (Polygon Formula) | Continuous metric of the object's surface. |


#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $H$ and $W$.
* Next $H$ lines: $W$ integers ($0$ or $1$) each.
* Last line: Integer $c$ ($4$ or $8$).

**Output:**

* One line per instance in discovery order:
`Instance l: Area=A, BBox=(x,y,w,h), Centroid=(cx,cy)`
* Last line: `Total instances: K`.



#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | Instance 1: Area=1.0, BBox=(1,1,2,2), Centroid=(1.50,1.50)<br>Instance 2: Area=1.0, BBox=(4,4,2,2), Centroid=(4.50,4.50)<br>Total instances: 2 | Aligned $2\times2$ blocks. The geometric contour area calculation results in $1.0$. The centroid of the block in columns 1–2 and rows 1–2 is exactly $(1.50,\,1.50)$. |
| 4 6<br>0 0 0 0 0 0<br>0 1 1 1 1 0<br>0 0 0 1 0 0<br>0 0 0 0 0 0<br>4 | Instance 1: Area=2.0, BBox=(1,1,4,2), Centroid=(2.40,1.20)<br>Total instances: 1 | Asymmetric inverted "T" shaped object. The geometric contour area is $2.0$. The centroid reflects the pixel distribution of the object. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0806" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulator EP08_06: Native Morphological Metrics (mm.measure)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">OpenCV Contour & Moments</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:14px;flex-wrap:wrap;gap:10px;">
      <div>
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ACTION</div>
        <button id="ep0806_btnRandom" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generate New Binary Instances</button>
      </div>
      <div style="font-size:11px;color:#8a8371;font-family:monospace;">
        <span style="font-weight:700;color:#26241d;">Precision Parameter (approxPolyDP):</span> precision = 0.01
      </div>
    </div>

    <!-- Container da Matriz de Píxeis -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">INSTANCE LABEL MAP</div>
      <div id="ep0806_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <!-- Tabela de Métricas do mm.measure -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">METRICS EXTRACTED BY MM.MEASURE</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">center (cx, cy)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0806_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0806Init) return;
    root.dataset.ep0806Init = "1";

    var H = 10, W = 22;
    var mask = [], labels = [], metrics = [];
    var colors = ['#ffffff', '#7ee7c6', '#fca5a5', '#fde047', '#93c5fd', '#c084fc', '#f472b6'];

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [];
            var queue = [[r, c]];
            visited[r][c] = true;

            while (queue.length > 0) {
              var curr = queue.shift();
              var cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);

              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true;
                    queue.push([nr, nc]);
                  }
                }
              }
            }

            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) {
                borderPts.push([pc, pr]);
              }
            });

            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;

            borderPts.sort(function(a, b) {
              return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx);
            });

            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1];
        area -= contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length;
      var m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }

      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else {
          return [pts[0], pts[end]];
        }
      }
      return rdp(contour, epsilon);
    }

    function measureJS(mat) {
      var blobs = cv2_findContours(mat);
      var res = [];
      labels = Array.from({length: H}, function(){ return Array(W).fill(0); });

      blobs.forEach(function(item, idx) {
        var contour = item.contour;
        var pixels = item.pixels;
        var labelId = idx + 1;

        pixels.forEach(function(p){ labels[p[0]][p[1]] = labelId; });

        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;

        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);

        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;

        var poly = cv2_approxPolyDP(contour, 0.01);

        res.push({
          id: labelId,
          area: area,
          perimeter: per,
          cx: moments.cx,
          cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0,
          vertices: poly.length
        });
      });

      res.sort(function(a, b) {
        if (a.y !== b.y) return a.y - b.y;
        return a.x - b.x;
      });

      res.forEach(function(m, i) { m.id = i + 1; });
      return res;
    }

    function render() {
      var gridContainer = root.querySelector('#ep0806_grid_container');
      gridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var l = labels[r][c];
          var cell = document.createElement('div');
          var bg = l === 0 ? '#ffffff' : colors[(l % (colors.length - 1)) + 1];
          var fg = l === 0 ? '#8a8371' : '#26241d';
          cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;font-family:monospace;user-select:none;background:' + bg + ';color:' + fg + ';';
          cell.textContent = l;
          grid.appendChild(cell);
        }
      }
      gridContainer.appendChild(grid);

      var tbody = root.querySelector('#ep0806_tbody');
      tbody.innerHTML = '';

      if (metrics.length === 0) {
        tbody.innerHTML = '<tr><td colspan="8" style="padding:12px;color:#8a8371;text-align:center;">Nenhuma instância binária encontrada.</td></tr>';
        return;
      }

      metrics.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        var centerStr = '(' + m.cx.toFixed(2) + ', ' + m.cy.toFixed(2) + ')';
        var bboxStr = '(' + m.x + ', ' + m.y + ', ' + m.w + ', ' + m.h + ')';

        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + centerStr + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxStr + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        tbody.appendChild(tr);
      });
    }

    function generate() {
      mask = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var numObj = Math.floor(Math.random() * 2) + 2;

      for (var o = 0; o < numObj; o++) {
        var w = Math.floor(Math.random() * 3) + 3;
        var h = Math.floor(Math.random() * 3) + 3;
        var sr = Math.floor(Math.random() * (H - h));
        var sc = Math.floor(Math.random() * (W / numObj - w)) + Math.floor(o * (W / numObj));

        for (var r = 0; r < h; r++) {
          for (var c = 0; c < w; c++) {
            if (Math.random() > 0.15) mask[sr + r][sc + c] = 1;
          }
        }
      }

      metrics = measureJS(mask);
      render();
    }

    root.querySelector('#ep0806_btnRandom').addEventListener('click', generate);
    generate();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0806');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.6:** Simulador EP08_06: Extraction of *Bounding Boxes*, Centroids and Properties with *mm.measure*


<figure id="fig-08-sim-ep0806">
  <img src="imagens/fig-08-sim-ep0806.png" alt=" Simulador EP08_06: Extraction of *Bounding Boxes*, Centroids and Properties with *mm.measure* " style="max-width:80%" />
  <figcaption><strong>Figure 8.6:</strong>  Simulador EP08_06: Extraction of *Bounding Boxes*, Centroids and Properties with *mm.measure* </figcaption>
</figure>

In [ ]:
%%writefile EP08_06.py
# Python code

In [ ]:
TestSuite("EP08_06.py").run()

### EP08_07 🟡 Salt-and-Pepper Noise Removal and Object Measurement

In this exercise, you will apply morphological filtering to clean a binary image corrupted by **salt-and-pepper** noise (isolated pixels of value `1` in the background and `0` inside objects). After cleaning, the program must extract the geometric measurements of the remaining connected components, sort them, and display the final metrics table.

#### 📋 Implementation Guidelines

1. **Input:** read two integers $H$ and $W$ (image height and width) from the first line, followed by $H$ lines containing the binary matrix with pixels `0` and `1` separated by spaces.

2. **Morphological Filtering:** apply a chain of **Opening** (to eliminate salt noise in the background) followed by **Closing** (to fill pepper noise inside objects) using a $3 \times 3$ structuring element.

3. **Printing the Cleaned Image:** print the resulting matrix with values `0` and `1` separated by spaces.

4. **Geometric Measurements:** for each object identified in the cleaned matrix, extract:
* `id`: sequential numeric identifier (reassigned after sorting);

* `area`: area calculated via contour (`cv2.contourArea`);

* `perimeter`: contour perimeter (`cv2.arcLength`);

* `cx`, `cy`: center of mass (centroid via `cv2.moments`);

* `x`, `y`, `w`, `h`: coordinates of the bounding rectangle (`cv2.boundingRect`);

* `circularity`: circularity given by $\frac{4 \pi \cdot \text{area}}{\text{perimeter}^2}$;
* `solidity`: solidity given by the ratio $\frac{\text{area}}{\text{convex hull area}}$;
* `vertices`: approximate number of polygon vertices (`cv2.approxPolyDP` with $\epsilon = 0.02 \times \text{perimeter}$).

5. **Sorting and Output:** sort objects in ascending order by the $X$ position of the bounding rectangle (`bbox[0]`); in case of a tie, use the $Y$ position (`bbox[1]`). Reassign `id`s from $1$ to $N$ and print the formatted table.
   - For sorting, use `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, with `medidas = mm.measure(img)`.

#### 📌 Constraints and Sorting Rules

* **Object Sorting Rule:**
```python
medidas.sort(key=lambda m: (m['bbox'][0], m['bbox'][1]))

```

* **Area Difference:** The area calculated by OpenCV (`cv2.contourArea`) measures the area of the continuous polygon delimited by the centers of border pixels, resulting in numeric values smaller than the simple discrete count of `1` pixels (`np.sum`).

#### 🧠 Theoretical Foundation

| Operation / Metric | Function in Filtering and Characterization |
|--------------------|---------------------------------------|
| **Morphological Opening** ($\circ$) | Erosion followed by dilation: removes isolated bright noise (*salt*). |
| **Morphological Closing** ($\bullet$) | Dilation followed by erosion: fills small dark holes inside objects (*pepper*). |
| **`cv2.boundingRect`** | Returns $(x, y, w, h)$, the smallest axis-aligned rectangle enclosing the object. |
| **Circularity and Solidity** | Describe the geometric compactness and convexity of the component. |

#### 📌 Examples

| Input | Output |
|---|---|
| 8 9<br>0 0 0 0 0 0 0 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 1 1<br>0 0 0 0 0 0 0 1 1<br>0 0 0 0 0 0 0 0 0 | id area perimeter cx cy x y w h circularity solidity vertices<br>1 9.0 12.0 3.5 2.0 3 1 4 3 0.79 1.000 4<br>2 4.0 8.0 7.5 5.5 7 5 2 2 0.79 1.000 4 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0807" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulator EP08_07: Switchable Morphology (4-C / 8-C) & OpenCV Metrics</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Salt + Pepper &rarr; Opening &rarr; Closing &rarr; Measurement</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:260px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MORPHOLOGICAL PROCESSING STAGE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnOrig" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Noisy</button>
          <button id="ep0807_btnAbert" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Opening</button>
          <button id="ep0807_btnFech" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Closing</button>
        </div>
      </div>

      <div style="flex:1;min-width:140px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">STRUCTURING ELEMENT</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnConn4" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">4-Conn.</button>
          <button id="ep0807_btnConn8" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">8-Conn.</button>
        </div>
      </div>

      <div style="min-width:140px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">PIXEL DISPLAY</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnVal" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">Values (0/1)</button>
          <button id="ep0807_btnCor" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">Colors (P&W)</button>
        </div>
      </div>

      <div>
        <button id="ep0807_btnRandom" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generate Random Scenario</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">INPUT / PROCESSED PIXEL MATRIX VIEW</div>
      <div id="ep0807_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">OBJECT MEASUREMENT TABLE (CALCULATED AFTER OPENING AND CLOSING)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0807_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0807Init) return;
    root.dataset.ep0807Init = "1";

    var H = 12, W = 28;
    var imgBase = [], imgRuido = [], imgAbertura = [], imgLimpa = [];
    var medidasObjetos = [];
    var etapaAtual = 'ruido', modoExibicao = 'val', modoConectividade = 4;

    var elBtnOrig = root.querySelector('#ep0807_btnOrig');
    var elBtnAbert = root.querySelector('#ep0807_btnAbert');
    var elBtnFech = root.querySelector('#ep0807_btnFech');
    var elBtnConn4 = root.querySelector('#ep0807_btnConn4');
    var elBtnConn8 = root.querySelector('#ep0807_btnConn8');
    var elBtnVal = root.querySelector('#ep0807_btnVal');
    var elBtnCor = root.querySelector('#ep0807_btnCor');
    var elBtnRandom = root.querySelector('#ep0807_btnRandom');
    var elGridContainer = root.querySelector('#ep0807_grid_container');
    var elTbody = root.querySelector('#ep0807_tbody');

    var neighbors4 = [[0,0], [-1,0], [1,0], [0,-1], [0,1]];
    var neighbors8 = [[0,0], [-1,0], [1,0], [0,-1], [0,1], [-1,-1], [-1,1], [1,-1], [1,1]];

    function dilate(mat, conn) {
      var neighbors = conn === 8 ? neighbors8 : neighbors4;
      var res = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var hit = false;
          for (var i = 0; i < neighbors.length; i++) {
            var nr = r + neighbors[i][0], nc = c + neighbors[i][1];
            if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
              if (mat[nr][nc] === 1) { hit = true; break; }
            }
          }
          res[r][c] = hit ? 1 : 0;
        }
      }
      return res;
    }

    function erode(mat, conn) {
      var neighbors = conn === 8 ? neighbors8 : neighbors4;
      var res = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var fit = true;
          for (var i = 0; i < neighbors.length; i++) {
            var nr = r + neighbors[i][0], nc = c + neighbors[i][1];
            if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
              if (mat[nr][nc] !== 1) { fit = false; break; }
            } else { fit = false; }
          }
          res[r][c] = fit ? 1 : 0;
        }
      }
      return res;
    }

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.01);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function recalcularMorfologia() {
      imgAbertura = dilate(erode(imgRuido, modoConectividade), modoConectividade);
      imgLimpa = erode(dilate(imgAbertura, modoConectividade), modoConectividade);
      medidasObjetos = measureOpenCV(imgLimpa);
      renderGrid();
      renderTabela();
    }

    function gerarCenarioAleatorio() {
      imgBase = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var numObjetos = Math.floor(Math.random() * 2) + 2; 
      var setores = [ { minC: 1, maxC: 8 }, { minC: 10, maxC: 17 }, { minC: 19, maxC: 25 } ];
      var objetoPixels = [];
      for (var o = 0; o < numObjetos; o++) {
        var setor = setores[o];
        var tipoForma = Math.floor(Math.random() * 3);
        var objW = Math.floor(Math.random() * 2) + 4, objH = Math.floor(Math.random() * 2) + 4;
        var startC = Math.floor(Math.random() * (setor.maxC - setor.minC - objW + 1)) + setor.minC;
        var startR = Math.floor(Math.random() * (H - 4 - objH + 1)) + 2;
        for (var r = 0; r < objH; r++) {
          for (var c = 0; c < objW; c++) {
            var pr = startR + r, pc = startC + c, isObj = false;
            if (tipoForma === 0) isObj = true;
            else if (tipoForma === 1) { if (r >= objH / 2 || c < objW / 2) isObj = true; }
            else if (tipoForma === 2) { if (r < objH / 2 || (c >= Math.floor(objW / 3) && c <= Math.floor(2 * objW / 3))) isObj = true; }
            if (isObj) { imgBase[pr][pc] = 1; objetoPixels.push([pr, pc]); }
          }
        }
      }
      imgRuido = JSON.parse(JSON.stringify(imgBase));
      var qtdSal = Math.floor(Math.random() * 2) + 2;
      for (var s = 0; s < qtdSal; s++) {
        var sr = Math.floor(Math.random() * (H - 2)) + 1, sc = Math.floor(Math.random() * (W - 2)) + 1;
        if (imgBase[sr][sc] === 0) imgRuido[sr][sc] = 1;
      }
      var shuffledObj = objetoPixels.filter(function(p){ return p[0] > 0 && p[0] < H-1 && p[1] > 0 && p[1] < W-1; }).sort(function() { return 0.5 - Math.random(); });
      var qtdPimenta = Math.max(1, Math.floor(shuffledObj.length * 0.10));
      for (var p = 0; p < qtdPimenta; p++) { imgRuido[shuffledObj[p][0]][shuffledObj[p][1]] = 0; }
      recalcularMorfologia();
    }

    function renderGrid(){
      var mat = imgRuido;
      if (etapaAtual === 'abertura') mat = imgAbertura;
      if (etapaAtual === 'fechamento') mat = imgLimpa;
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var val = mat[r][c], cell = document.createElement('div');
          var bg = val === 1 ? '#26241d' : '#ffffff';
          var fg = val === 1 ? '#7ee7c6' : '#8a8371';
          cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;font-family:monospace;user-select:none;background:' + bg + ';color:' + fg + ';';
          if (modoExibicao === 'val') cell.textContent = val; else cell.textContent = '';
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o processamento morfológico.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setEtapa(etapa, btn){
      [elBtnOrig, elBtnAbert, elBtnFech].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      etapaAtual = etapa; renderGrid();
    }

    function setConectividade(conn, btn){
      [elBtnConn4, elBtnConn8].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      modoConectividade = conn; recalcularMorfologia();
    }

    function setModo(modo, btn){
      [elBtnVal, elBtnCor].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      modoExibicao = modo; renderGrid();
    }

    elBtnOrig.addEventListener('click', function(){ setEtapa('ruido', elBtnOrig); });
    elBtnAbert.addEventListener('click', function(){ setEtapa('abertura', elBtnAbert); });
    elBtnFech.addEventListener('click', function(){ setEtapa('fechamento', elBtnFech); });
    elBtnConn4.addEventListener('click', function(){ setConectividade(4, elBtnConn4); });
    elBtnConn8.addEventListener('click', function(){ setConectividade(8, elBtnConn8); });
    elBtnVal.addEventListener('click', function(){ setModo('val', elBtnVal); });
    elBtnCor.addEventListener('click', function(){ setModo('cor', elBtnCor); });
    elBtnRandom.addEventListener('click', function(){ gerarCenarioAleatorio(); });

    gerarCenarioAleatorio();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0807');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.7:** EP08_07 Simulator: Morphology with Configurable Connectivity and Measurement


<figure id="fig-08-sim-ep0807">
  <img src="imagens/fig-08-sim-ep0807.png" alt=" EP08_07 Simulator: Morphology with Configurable Connectivity and Measurement " style="max-width:80%" />
  <figcaption><strong>Figure 8.7:</strong>  EP08_07 Simulator: Morphology with Configurable Connectivity and Measurement </figcaption>
</figure>

In [ ]:
%%writefile EP08_07.py
# Python code

In [ ]:
TestSuite("EP08_07.py").run()

### EP08_08 🟡 Grayscale Image and Dynamic Thresholding

In this exercise, the input image is no longer strictly binary (`0`/`1`) but becomes a **grayscale image ($8$ bits, $0\dots255$)**, where objects have an intermediate average intensity over a dark background ($0$), in addition to salt-and-pepper noise scattered throughout the entire image.

#### 📋 Implementation Guidelines

1. **Input:** read $H$ and $W$ on the first line, followed by the $H$ lines with integer values from $0$ to $255$ in an $H \times W$ matrix.
2. **Preprocessing:**
* Apply a **Median filter ($3 \times 3$)** to remove salt-and-pepper noise while keeping edges sharp.
* Apply **Otsu's thresholding** (or a fixed threshold $T = 60$) to binarize the clean image.


3. **Measurement and Output:** extract the contour of the objects, compute the geometric metrics (`area`, `perimeter`, `cx`, `cy`, `x`, `y`, `w`, `h`, `circularity`, `solidity`), and sort the objects by `bbox[0]` (with `bbox[1]` as a tiebreaker). Reassign `id` from $1$ to $N$ and print the table.
   - For sorting, use `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, with `medidas = mm.measure(img)`.


#### 📌 Examples

| Input | Output |
|---|---|
| 16 32<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>(binary image containing a square and a circle) | id area perimeter cx cy x y w h circularity solidity vertices<br>1 16.0 16.0 8.0 8.0 6 6 5 5 0.79 1.000 4<br>2 28.3 18.8 22.5 8.0 19 5 7 7 1.00 1.000 8 |

\newpage

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0808" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulator EP08_08: Salt and Pepper Noise in Grayscale & OpenCV Measurement</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Median 3x3 &rarr; Binarization &rarr; Measurement</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:260px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">PROCESSING STAGE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0808_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Noisy Grayscale</button>
          <button id="ep0808_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Median 3x3</button>
          <button id="ep0808_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Binarized (Otsu)</button>
        </div>
      </div>

      <div>
        <button id="ep0808_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generate Random Shapes/Positions</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">PIXEL MATRIX VIEW</div>
      <div id="ep0808_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">OBJECT MEASUREMENT TABLE (SORTED BY BBOX_X, BBOX_Y)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0808_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0808Init) return;
    root.dataset.ep0808Init = "1";

    var H = 10, W = 20, stage = 0;
    var matOrig = [], matMed = [], matBin = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0808_btnRand');
    var elGridContainer = root.querySelector('#ep0808_grid_container');
    var elTbody = root.querySelector('#ep0808_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var w1 = Math.floor(Math.random() * 2) + 4, h1 = Math.floor(Math.random() * 2) + 4;
      var x1 = Math.floor(Math.random() * 2) + 1, y1 = Math.floor(Math.random() * 2) + 1;
      var val1 = 150;
      for (var r = y1; r < y1 + h1; r++) { for (var c = x1; c < x1 + w1; c++) matOrig[r][c] = val1; }

      var w2 = Math.floor(Math.random() * 2) + 4, h2 = Math.floor(Math.random() * 2) + 4;
      var x2 = Math.floor(Math.random() * 2) + 11, y2 = Math.floor(Math.random() * 2) + 2;
      var val2 = 180;
      for (var r = y2; r < y2 + h2; r++) { for (var c = x2; c < x2 + w2; c++) matOrig[r][c] = val2; }

      for (var i = 0; i < 4; i++) {
        var sr = Math.floor(Math.random() * H), sc = Math.floor(Math.random() * W);
        if (matOrig[sr][sc] === 0) matOrig[sr][sc] = 255;
      }
      matOrig[y1 + 1][x1 + 1] = 0; matOrig[y2 + 1][x2 + 1] = 0;

      matMed = JSON.parse(JSON.stringify(matOrig));
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var vals = [];
          for (var dr = -1; dr <= 1; dr++) {
            for (var dc = -1; dc <= 1; dc++) {
              var nr = r + dr, nc = c + dc;
              if (nr >= 0 && nr < H && nc >= 0 && nc < W) { vals.push(matOrig[nr][nc]); } else { vals.push(0); }
            }
          }
          vals.sort(function(a, b){ return a - b; });
          matMed[r][c] = vals[4];
        }
      }

      matBin = matMed.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      medidasObjetos = measureOpenCV(matBin);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matOrig : (stage === 1 ? matMed : matBin);

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          if (stage === 2) {
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else {
            var fgCinza = v > 128 ? '#000000' : '#ffffff';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + v + ',' + v + ',' + v + ');color:' + fgCinza + ';';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após a filtragem.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0808_stage0'), root.querySelector('#ep0808_stage1'), root.querySelector('#ep0808_stage2')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0808_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0808_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0808_stage2').addEventListener('click', function(){ setStage(2, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0808');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.8:** EP08_08 Simulator: Median Filtering in Grayscale and Object Measurement with OpenCV


<figure id="fig-08-sim-ep0808">
  <img src="imagens/fig-08-sim-ep0808.png" alt=" EP08_08 Simulator: Median Filtering in Grayscale and Object Measurement with OpenCV " style="max-width:80%" />
  <figcaption><strong>Figure 8.8:</strong>  EP08_08 Simulator: Median Filtering in Grayscale and Object Measurement with OpenCV </figcaption>
</figure>

In [ ]:
%%writefile EP08_08.py
# Python code

In [ ]:
TestSuite("EP08_08.py").run()

### EP08_09 🟠 Illumination Gradient and Adaptive Thresholding

In this variation, the objects are immersed in a background with **non-uniform illumination (smooth illumination gradient)**. Simple single-value thresholding fails, requiring more robust preprocessing.

#### 📋 Implementation Guidelines

1. **Input:** grayscale image $H \times W$ with background variation from $20$ to $180$.
2. **Preprocessing:**
  
* Apply **Adaptive Thresholding** (e.g., `cv2.adaptiveThreshold` with a Gaussian window of $15 \times 15$ and constant $C = 3$) to isolate the objects regardless of background variation.
  
  `cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, ksize, C) // 255`

  `ksize` and `C` are read after the image.
  
* Morphological **Closing** operation ($3 \times 3$) to seal any gaps in the contours.


1. **Measurement and Classification:** extract the measurements.


4. **Sorting and Output:** sort by `(bbox[0], bbox[1])` and print the table including the `class` column.
   - To sort, use `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, with `medidas = mm.measure(img, precision=0.02)`.



#### 📌 Examples

| Input | Output |
|---|---|
| 16 32<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>k 20 | id area perimeter cx cy x y w h circularity solidity vertices<br>1 9.0 12.0 10.0 5.0 8 3 5 5 0.79 1.000 4<br>2 28.3 18.8 25.0 12.0 22 9 7 7 1.00 1.000 3|

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0809" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulator EP08_09: Illumination Gradient and Adaptive Threshold & OpenCV Measurement</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Adaptive vs Global &rarr; Measurement</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:280px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">PROCESSING STAGE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0809_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Grayscale Gradient</button>
          <button id="ep0809_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Global Threshold Fails</button>
          <button id="ep0809_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Adaptive Threshold OK</button>
        </div>
      </div>

      <div>
        <button id="ep0809_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generate Random Shapes/Positions</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">PIXEL MATRIX VIEW</div>
      <div id="ep0809_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">OBJECT MEASUREMENT TABLE (COMPUTED AT ADAPTIVE THRESHOLD OK)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0809_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0809Init) return;
    root.dataset.ep0809Init = "1";

    var H = 10, W = 20, stage = 0;
    var matGrad = [], matGlob = [], matAdapt = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0809_btnRand');
    var elGridContainer = root.querySelector('#ep0809_grid_container');
    var elTbody = root.querySelector('#ep0809_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matGrad = Array.from({length: H}, function(_, r){
        return Array.from({length: W}, function(_, c){ return Math.round(20 + c * 9.5); });
      });
      var w1 = 3, h1 = 3;
      var x1 = Math.floor(Math.random() * 2) + 2, y1 = Math.floor(Math.random() * 2) + 2;
      for (var r = y1; r < y1 + h1; r++) { for (var c = x1; c < x1 + w1; c++) matGrad[r][c] += 90; }

      var w2 = 3, h2 = 3;
      var x2 = Math.floor(Math.random() * 2) + 14, y2 = Math.floor(Math.random() * 2) + 2;
      for (var r = y2; r < y2 + h2; r++) { for (var c = x2; c < x2 + w2; c++) matGrad[r][c] += 90; }

      matGlob = matGrad.map(function(row) { return row.map(function(v) { return v > 110 ? 1 : 0; }); });

      var blockSize = 5, half = Math.floor(blockSize / 2), C_val = 20;
      matAdapt = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var sum = 0, count = 0;
          for (var dr = -half; dr <= half; dr++) {
            for (var dc = -half; dc <= half; dc++) {
              var nr = r + dr, nc = c + dc;
              if (nr >= 0 && nr < H && nc >= 0 && nc < W) { sum += matGrad[nr][nc]; count++; }
            }
          }
          var mean = sum / count;
          matAdapt[r][c] = matGrad[r][c] > (mean + C_val) ? 1 : 0;
        }
      }
      medidasObjetos = measureOpenCV(matAdapt);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matGrad : (stage === 1 ? matGlob : matAdapt);

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          if (stage > 0) {
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else {
            var fgCinza = v > 120 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o limiar adaptativo.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0809_stage0'), root.querySelector('#ep0809_stage1'), root.querySelector('#ep0809_stage2')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0809_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0809_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0809_stage2').addEventListener('click', function(){ setStage(2, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0809');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.9:** Simulator EP08_09: Illumination Gradient, Adaptive Threshold and OpenCV Measurement


<figure id="fig-08-sim-ep0809">
  <img src="imagens/fig-08-sim-ep0809.png" alt=" Simulator EP08_09: Illumination Gradient, Adaptive Threshold and OpenCV Measurement " style="max-width:80%" />
  <figcaption><strong>Figure 8.9:</strong>  Simulator EP08_09: Illumination Gradient, Adaptive Threshold and OpenCV Measurement </figcaption>
</figure>

In [ ]:
%%writefile EP08_09.py
# Python code

In [ ]:
TestSuite("EP08_09.py").run()

### EP08_10 🔴 Low Contrast and Object Separation Topics (*Watershed* / Distance)

In this exercise, **some geometric objects are slightly touching (overlapping at the edges)**. Simply extracting contours would treat two objects as one.

#### 📋 Implementation Guidelines

1. **Input:** an $H \times W$ grayscale matrix with objects of intensity $110\dots140$ on a background of $0$, with noise and a pair of tangent objects.
2. **Preprocessing and Separation:**
* Apply thresholding.
* Apply the **Distance Transform** (`mm.dist`).
* Obtain distance peaks to serve as markers in the **Watershed Transform** (`mm.watershed`), physically separating the touching objects in the mask. **Hint:** use `mm.regmax()` to obtain the local maxima and then label them with `mm.label0`.
* After the watershed, apply thresholding again with `mm.threshold(water,0)//255`.

3. **Connected Component Analysis:** measure each isolated region after the watershed.
4. **Output:** print the components sorted by `(bbox[0], bbox[1])` with their individual area, centroid, and solidity metrics.
   - For sorting, use `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, with `medidas = mm.measure(img, precision=0.02)`.

#### 📌 Examples

| Input | Output |
|---|---|
| 16 16<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>(binary image containing two squares) | id area perimeter cx cy x y w h solidity<br>1 16.0 16.0 5.0 5.0 3 3 5 5 1.000<br>2 16.0 16.0 11.0 5.0 9 3 5 5 1.000 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0810" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulator EP08_10: Separation of Tangent Disks (L2 Transform & Watershed)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">mm.dist L2 &rarr; mm.watershed &rarr; Measurement</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:300px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MORPHOLOGICAL PROCESSING STEP</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0810_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Noisy Gray</button>
          <button id="ep0810_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Joined Mask</button>
          <button id="ep0810_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. L2 Distance</button>
          <button id="ep0810_stage3" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">4. Watershed (Cut)</button>
        </div>
      </div>

      <div>
        <button id="ep0810_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generate Disks with Random Radii</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">PIXEL MATRIX VISUALIZATION</div>
      <div id="ep0810_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">DISK MEASUREMENTS TABLE AFTER WATERSHED CUT</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0810_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0810Init) return;
    root.dataset.ep0810Init = "1";

    var H = 11, W = 21, stage = 0;
    var matOrig = [], matBin = [], matDist = [], matWash = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0810_btnRand');
    var elGridContainer = root.querySelector('#ep0810_grid_container');
    var elTbody = root.querySelector('#ep0810_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var r1 = Math.floor(Math.random() * 3) + 2, r2 = Math.floor(Math.random() * 3) + 2; 
      var cy1 = Math.floor(Math.random() * 2) + 4, cx1 = Math.floor(Math.random() * 2) + 3;
      var cx2 = cx1 + r1 + r2, cy2 = cy1;

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var d1 = Math.hypot(r - cy1, c - cx1), d2 = Math.hypot(r - cy2, c - cx2);
          if (d1 <= r1 || d2 <= r2) { matOrig[r][c] = 140 + Math.floor(Math.random() * 20); }
        }
      }

      matBin = matOrig.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      matDist = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var fundoPixels = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) { if (matBin[r][c] === 0) fundoPixels.push([r, c]); }
      }

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (matBin[r][c] === 1) {
            var minDist = Infinity;
            for (var k = 0; k < fundoPixels.length; k++) {
              var dist = Math.hypot(r - fundoPixels[k][0], c - fundoPixels[k][1]);
              if (dist < minDist) minDist = dist;
            }
            matDist[r][c] = Math.round(minDist);
          }
        }
      }

      matWash = JSON.parse(JSON.stringify(matBin));
      var colCorte = cx1 + r1; 
      for (var r = 0; r < H; r++) { if (matWash[r][colCorte] === 1) matWash[r][colCorte] = 0; }

      medidasObjetos = measureOpenCV(matWash);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var cell = document.createElement('div');
          if (stage === 0) {
            var v = matOrig[r][c]; cell.textContent = v;
            var fgCinza = v > 120 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';';
          } else if (stage === 1) {
            var v = matBin[r][c]; cell.textContent = v;
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else if (stage === 2) {
            var v = matDist[r][c]; cell.textContent = v;
            var bgDist = v > 0 ? 'rgb(' + (240 - v * 45) + ',' + (240 - v * 30) + ',255)' : '#ffffff';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgDist + ';color:#26241d;';
          } else {
            var v = matWash[r][c]; cell.textContent = v;
            var bgWash = v === 1 ? '#26241d' : '#ffffff', fgWash = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgWash + ';color:' + fgWash + ';';
          }
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o corte do Watershed.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0810_stage0'), root.querySelector('#ep0810_stage1'), root.querySelector('#ep0810_stage2'), root.querySelector('#ep0810_stage3')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0810_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0810_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0810_stage2').addEventListener('click', function(){ setStage(2, this); });
    root.querySelector('#ep0810_stage3').addEventListener('click', function(){ setStage(3, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0810');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.10:** EP08_10 Simulator: Separation of Tangent Disks via L2 Distance Transform and *Watershed*


<figure id="fig-08-sim-ep0810">
  <img src="imagens/fig-08-sim-ep0810.png" alt=" EP08_10 Simulator: Separation of Tangent Disks via L2 Distance Transform and *Watershed* " style="max-width:80%" />
  <figcaption><strong>Figure 8.10:</strong>  EP08_10 Simulator: Separation of Tangent Disks via L2 Distance Transform and *Watershed* </figcaption>
</figure>

\newpage

In [ ]:
%%writefile EP08_10.py
# Python code

In [ ]:
TestSuite("EP08_10.py").run()

### EP08_11 🔴 Classification and Validation of Objects with Bounding Box Ground Truth

In this exercise, the goal is to process a grayscale image containing multiple geometric objects, extract their properties using `mm.measure`, and validate the detected bounding boxes against a real ground truth (GT) provided as input, using the IoU (Intersection over Union) metric.

#### 📋 Implementation Guidelines

1. **Image Reading:** Read the dimensions $H \times W$ and the $H \times W$ pixel matrix of the grayscale image.
2. ***Morphological Pipeline:*** Binarize the image using the Otsu method (`mm.threshold`) and render the resulting binarized mask using `mm.drawImg`.
3. **Ground Truth Reading:**
   
   * Read the number $G$ of ground truth bounding boxes.
   * If $G > 0$, read $G$ lines each containing 5 values: `id xmin_norm ymin_norm xmax_norm ymax_norm`.
   * **Coordinate Conversion:** The ground truth coordinates are normalized in the range $[0.0, 1.0]$. To convert them to pixels on the image grid:

$$x_{\min} = \lfloor \text{xmin\_norm} \times W \rfloor, \quad y_{\min} = \lfloor \text{ymin\_norm} \times H \rfloor$$

$$w = \lfloor \text{xmax\_norm} \times W \rfloor - x_{\min}, \quad h = \lfloor \text{ymax\_norm} \times H \rfloor - y_{\min}$$

4. **Metric Extraction and IoU Calculation:**
   * Extract instance properties using `mm.measure(img_bin, precision=0.02)`.
   * For each detected bounding box $(x, y, w, h)$, compute the IoU overlap with respect to the ground truth boxes and set `hits = 1` if there exists any match with $\text{IoU} \ge 0.50$, otherwise set `hits = 0`.

5. **Output:** Sort the instances by position `(bbox[0], bbox[1])` and print the CSV table with the additional column `hits`.
   - For sorting, use `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, with `medidas = mm.measure(img)`.

#### 🧠 Theoretical Foundation and Conversion

| Concept | Formula / Operation | Description |
| --- | --- | --- |
| **Detected BBox** | $(x, y, w, h)$ via `mm.measure` | Bounding box computed on the discrete grid in integer pixels. |
| **Ground Truth BBox (GT)** | $(x_{\min}, y_{\min}, w, h)$ converted | Real box provided as input in relative coordinates $[0.0, 1.0]$. |
| **IoU (Intersection over Union)** | $\text{IoU} = \frac{\text{Area}(B_{\text{DET}} \cap B_{\text{GT}})}{\text{Area}(B_{\text{DET}} \cup B_{\text{GT}})}$ | Evaluates the overlap rate of the boxes. It is considered valid if $\text{IoU} \ge 0.50$. |
| **Validation Status (`hits`)** | $1$ if $\max(\text{IoU}) \ge 0.50$, otherwise $0$ | Binary indicator of detector correctness relative to the ground truth. |

#### 📦 Input and Output Specification (VPL)

**Input:**

* **Line 1:** Integers $H$ and $W$ (dimensions of the matrix).
* **Next $H$ lines:** $W$ integers ($0$ to $255$) representing the grayscale image.
* **Line $H + 2$:** Integer $G$ (number of true ground truth boxes).
* **Next $G$ lines:** 5 numeric values per line: `id xmin_norm ymin_norm xmax_norm ymax_norm` (where the coordinates are floating-point values between $0.0$ and $1.0$).

**Output:**

1. Rendered binarized matrix via `mm.drawImg(img_bin)`.
2. CSV header: `id,area,perimeter,cx,cy,x,y,w,h,circularity,solidity,vertices,hits`
3. One CSV line per detected object containing its formatted properties and the `hits` indicator ($1$ or $0$).

#### 📌 Examples

| Input | Output |
|---|---|
| 10 20<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 180 0 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 180 180 180 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 0 180 0 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>2<br>1 0.10 0.30 0.25 0.60<br>2 0.60 0.30 0.75 0.60 | 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 1 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>id area perimeter cx cy x y w h circularity solidity vertices hits<br>1 2.0 5.7 3.0 4.0 2 3 3 3 0.79 1.000 4 1<br>2 4.0 8.0 13.0 4.0 12 3 3 3 0.79 1.000 4 1 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0811" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulator EP08_11: Bounding Boxes and IoU Comparison with Independent Controls</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">GT vs DET BBox Validation</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:1.5;min-width:220px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">DISPLAY MODE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0811_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Original Grayscale</button>
          <button id="ep0811_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Binarized + Overlays</button>
        </div>
      </div>

      <div style="flex:1.5;min-width:240px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">BOUNDING BOX DISPLAY</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0811_toggleGT" style="padding:5px 10px;font-size:10.5px;font-weight:700;border:1px solid #1d4ed8;background:#2563eb;color:#ffffff;cursor:pointer;border-radius:8px;white-space:nowrap;display:inline-flex;align-items:center;gap:5px;"><span>🟦</span> Ground Truth BBox (GT)</button>
          <button id="ep0811_toggleDET" style="padding:5px 10px;font-size:10.5px;font-weight:700;border:1px solid #047857;background:#059669;color:#ffffff;cursor:pointer;border-radius:8px;white-space:nowrap;display:inline-flex;align-items:center;gap:5px;"><span>🟩</span> Detected BBox (DET)</button>
        </div>
      </div>

      <div>
        <button id="ep0811_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generate Random Scenes</button>
      </div>
    </div>

    <div style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:10px;padding:8px 12px;margin-bottom:12px;display:flex;gap:16px;flex-wrap:wrap;align-items:center;justify-content:center;">
      <span style="font-size:9.5px;font-weight:700;color:#8a8371;margin-right:4px;">BBOX LEGEND:</span>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#2563eb;border:1px dashed #93c5fd;"></span> <span>Real Ground Truth (GT)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#059669;border:1px solid #34d399;"></span> <span>Accepted Detection (IoU &ge; 0.5)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#dc2626;border:1px solid #f87171;"></span> <span>Rejected Detection (IoU &lt; 0.5)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#7c3aed;border:1px double #a78bfa;"></span> <span>BBox Overlap</span></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">PIXEL MATRIX VIEW</div>
      <div id="ep0811_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MEASUREMENTS, GEOMETRIC CLASSIFICATION AND IoU COMPARISON WITH GROUND TRUTH</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">class</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox det (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox gt (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">IoU</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">status (IoU &ge; 0.5)</th>
            </tr>
          </thead>
          <tbody id="ep0811_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0811Init) return;
    root.dataset.ep0811Init = "1";

    var H = 10, W = 20, stage = 0;
    var showGT = true, showDET = true;
    // matOrig/matBin: cena. medidasObjetos: 1 registro por objeto real (GT), casado com sua melhor DET.
    // detBoxesAtuais: caixas "detectadas" simuladas (com ruído/deslocamento em relação ao objeto real).
    var matOrig = [], matBin = [], medidasObjetos = [], detBoxesAtuais = [];

    var elBtnRand = root.querySelector('#ep0811_btnRand');
    var elToggleGT = root.querySelector('#ep0811_toggleGT');
    var elToggleDET = root.querySelector('#ep0811_toggleDET');
    var elGridContainer = root.querySelector('#ep0811_grid_container');
    var elTbody = root.querySelector('#ep0811_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function calculateIoU(boxA, boxB) {
      var ax1 = boxA.x, ay1 = boxA.y, ax2 = boxA.x + boxA.w, ay2 = boxA.y + boxA.h;
      var bx1 = boxB.x, by1 = boxB.y, bx2 = boxB.x + boxB.w, by2 = boxB.y + boxB.h;
      var ix1 = Math.max(ax1, bx1), iy1 = Math.max(ay1, by1);
      var ix2 = Math.min(ax2, bx2), iy2 = Math.min(ay2, by2);
      var iw = Math.max(0, ix2 - ix1), ih = Math.max(0, iy2 - iy1);
      var inter = iw * ih;
      var areaA = boxA.w * boxA.h, areaB = boxB.w * boxB.h;
      var union = areaA + areaB - inter;
      return union > 0 ? inter / union : 0.0;
    }

    // FIX: bbox extraída dos pixels reais do objeto (mat) é o GABARITO (GT) — é a posição
    // verdadeira e exata do objeto na cena. As caixas em `detBoxes` (com deslocamento
    // aleatório) representam a saída ruidosa de um detector, e são casadas ao GT mais
    // próximo por IoU.
    function measureOpenCV(mat, detBoxes) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var gtBbox = cv2_boundingRect(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        var solidity = hull_area > 0 ? area / hull_area : 0;
        var classe = solidity < 0.85 ? "Cruz (Côncavo)" : "Retângulo (Convexo)";
        var bestIoU = 0.0, matchedDET = { x: 0, y: 0, w: 0, h: 0 };

        detBoxes.forEach(function(d) {
          var iou = calculateIoU(gtBbox, d);
          if (iou > bestIoU) { bestIoU = iou; matchedDET = d; }
        });

        medidas.push({
          classe: classe, area: area, gtBox: gtBbox, detBox: matchedDET,
          solidity: solidity, vertices: poly.length, iou: bestIoU, ok: bestIoU >= 0.50
        });
      });
      medidas.sort(function(a, b) { return a.gtBox.x !== b.gtBox.x ? a.gtBox.x - b.gtBox.x : a.gtBox.y - b.gtBox.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var armLen = Math.floor(Math.random() * 2) + 1, thick = 1;
      var w1 = armLen * 2 + thick, h1 = armLen * 2 + thick;
      var x1 = Math.floor(Math.random() * Math.max(1, 8 - w1)) + 1;
      var y1 = Math.floor(Math.random() * Math.max(1, H - h1)) + 1;

      for (var r = 0; r < h1; r++) {
        for (var c = 0; c < w1; c++) {
          if ((c >= armLen && c < armLen + thick) || (r >= armLen && r < armLen + thick)) { matOrig[y1 + r][x1 + c] = 180; }
        }
      }

      var w2 = Math.floor(Math.random() * 3) + 3, h2 = Math.floor(Math.random() * 3) + 3;
      var x2 = Math.floor(Math.random() * Math.max(1, W - 10 - w2)) + 10;
      var y2 = Math.floor(Math.random() * Math.max(1, H - h2)) + 1;

      for (var r = y2; r < y2 + h2; r++) {
        for (var c = x2; c < x2 + w2; c++) { matOrig[r][c] = 180; }
      }

      // Estas caixas simulam a saída de um DETECTOR real: deslocadas/imprecisas em relação
      // ao objeto verdadeiro (que será obtido depois via segmentação em matBin -> GT).
      var shiftX1 = Math.random() > 0.5 ? 1 : 0, shiftY1 = Math.random() > 0.5 ? 1 : 0;
      var shiftX2 = Math.random() > 0.6 ? -2 : 0;

      detBoxesAtuais = [
        { x: Math.max(0, x1 + shiftX1), y: Math.max(0, y1 + shiftY1), w: w1, h: h1 },
        { x: Math.max(0, x2 + shiftX2), y: y2, w: w2 + (shiftX2 !== 0 ? 2 : 0), h: h2 }
      ];

      matBin = matOrig.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      medidasObjetos = measureOpenCV(matBin, detBoxesAtuais);
      renderGrid();
      renderTabela();
    }

    function inBox(r, c, box) { return r >= box.y && r < box.y + box.h && c >= box.x && c < box.x + box.w; }
    function isBoxEdge(r, c, box) { if (!inBox(r, c, box)) return false; return r === box.y || r === box.y + box.h - 1 || c === box.x || c === box.x + box.w - 1; }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matOrig : matBin;

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          var isGT = false, isDetOK = false, isDetFail = false;

          if (stage === 1) {
            // FIX: GT agora vem de m.gtBox (bbox real extraída dos pixels do objeto)
            if (showGT) { medidasObjetos.forEach(function(m) { if (isBoxEdge(r, c, m.gtBox)) isGT = true; }); }
            // FIX: DET agora vem de m.detBox (bbox ruidosa casada por IoU)
            if (showDET) {
              medidasObjetos.forEach(function(m) {
                if (isBoxEdge(r, c, m.detBox)) { if (m.ok) isDetOK = true; else isDetFail = true; }
              });
            }

            if (isGT && isDetOK) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#7c3aed;color:#ffffff;border:2px double #a78bfa;box-sizing:border-box;';
            } else if (isGT && isDetFail) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#c026d3;color:#ffffff;border:2px double #f472b6;box-sizing:border-box;';
            } else if (isDetOK) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#059669;color:#ffffff;border:2px solid #34d399;box-sizing:border-box;';
            } else if (isDetFail) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#dc2626;color:#ffffff;border:2px solid #f87171;box-sizing:border-box;';
            } else if (isGT) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#2563eb;color:#ffffff;border:2px dashed #93c5fd;box-sizing:border-box;';
            } else {
              var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';border:none;';
            }
          } else {
            var fgCinza = v > 100 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';border:none;';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="9" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado na cena.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        // FIX: bboxDet vem de m.detBox (caixa ruidosa) e bboxGT vem de m.gtBox (caixa real)
        var bboxDet = '(' + m.detBox.x + ',' + m.detBox.y + ',' + m.detBox.w + ',' + m.detBox.h + ')';
        var bboxGT = '(' + m.gtBox.x + ',' + m.gtBox.y + ',' + m.gtBox.w + ',' + m.gtBox.h + ')';
        var statusHtml = m.ok ? '<span style="color:#27ae60;font-weight:bold;">✔ True</span>' : '<span style="color:#e74c3c;font-weight:bold;">✖ False</span>';

        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.classe + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxDet + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxGT + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.iou.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + statusHtml + '</td>';
        elTbody.appendChild(tr);
      });
    }

    elToggleGT.addEventListener('click', function(){
      showGT = !showGT;
      if (showGT) {
        this.style.background = '#2563eb'; this.style.borderColor = '#1d4ed8'; this.style.color = '#ffffff';
        this.querySelector('span').textContent = '🟦';
      } else {
        this.style.background = '#f1ead7'; this.style.borderColor = '#d4cebe'; this.style.color = '#5e5a4a';
        this.querySelector('span').textContent = '⬜';
      }
      renderGrid();
    });

    elToggleDET.addEventListener('click', function(){
      showDET = !showDET;
      if (showDET) {
        this.style.background = '#059669'; this.style.borderColor = '#047857'; this.style.color = '#ffffff';
        this.querySelector('span').textContent = '🟩';
      } else {
        this.style.background = '#f1ead7'; this.style.borderColor = '#d4cebe'; this.style.color = '#5e5a4a';
        this.querySelector('span').textContent = '⬜';
      }
      renderGrid();
    });

    elBtnRand.addEventListener('click', gerarCenario);

    function setStage(s, btn){
      [root.querySelector('#ep0811_stage0'), root.querySelector('#ep0811_stage1')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    root.querySelector('#ep0811_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0811_stage1').addEventListener('click', function(){ setStage(1, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0811');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.11:** EP08_11 Simulator: Geometric Classification with Independent Controls for *BBox Overlays* (GT and DET)


<figure id="fig-08-sim-ep0811">
  <img src="imagens/fig-08-sim-ep0811.png" alt=" EP08_11 Simulator: Geometric Classification with Independent Controls for *BBox Overlays* (GT and DET) " style="max-width:80%" />
  <figcaption><strong>Figure 8.11:</strong>  EP08_11 Simulator: Geometric Classification with Independent Controls for *BBox Overlays* (GT and DET) </figcaption>
</figure>

In [ ]:
%%writefile EP08_11.py
# Python code

In [ ]:
TestSuite("EP08_11.py").run()

### EP08_12 🔴 Instance Segmentation on Real Images: Geometric Objects

The classic segmentation example in this chapter separated coin "instances" by spatial disconnection in the binary mask resulting from Otsu's thresholding. In this exercise, you will apply the same idea—but now on a real image with varied geometric objects—by chaining together preprocessing, binarization, contour extraction (`cv2.findContours`), and validation of the result against a ground truth of bounding boxes.

Unlike the previous exercise (labeling on an already-prepared mask), here you start from the **original image**: the quality of your segmentation depends directly on the preprocessing choices (filtering, thresholding, morphological operations) made before labeling the components.

#### 📋 Implementation Guidelines

1. **Input:** use the image `00000.jpg`.
2. **Preprocessing and segmentation:** apply the necessary steps (filtering, binarization, and morphological operations) to automatically separate the objects from the background, without manual cropping.
3. **Labeling and measurement:** for each segmented object, determine:
   - area;
   - center of mass (centroid);
   - type, according to the `obj2` set.
4. **Visual annotation:** write, inside each object, its area and the type abbreviation (`obj2`).
5. **Validation (IoU):** compute the Intersection over Union (IoU) between the detected bounding box (`cv2.boundingRect`) and the ground truth bounding box of the corresponding type. An object is considered correctly segmented only if there is exactly one bounding box of the correct type with **IoU ≥ 0.5**.
6. **Output:** print, for each detected object, its identifier, type, and whether it was successfully validated (`acertou=1`) or not. The output must follow the order of the `obj2` classes (0=Tria … 8=Cruz); within the same class, sort the objects by the vertical coordinate of the centroid (`cy`) in increasing order. Finally, print the overall accuracy.

#### 📌 Computational Constraints

* **No manual cropping:** all segmentation must be performed on the full image.
* **Fixed set of classes:**
  ```python
  obj  = ['Triangulo','Quadrado','Pendagono','Hexagono','Heptagono','Circulo',
          'Elipse','Estrela','Cruz']
  obj2 = ['Tria','Quad','Pent','Hexa','Hept','Circ','Elip','Estr','Cruz']
  ```
* **Image dimensions:** 608 × 608 pixels—used to denormalize the coordinates from the TXT file.
* **Center of mass validation:** an object is only considered correctly segmented if its centroid lies strictly inside the ground truth bounding box corresponding to the same object type.

#### 🧠 Theoretical Background

| Element | Role in instance segmentation |
|---|---|
| Preprocessing (filtering, thresholding) | Stage that produces the binary mask from the original intensity image |
| `cv2.findContours` | Extracts the contours of connected components in the binary mask |
| Geometric moments (`cv2.moments`) | Allow computing the center of mass (centroid) of each contour |
| `approxPolyDP` / vertices | Aids in classifying the object type (approximate number of sides) |
| Bounding box validation | Confirms whether the segmented instance spatially corresponds to a ground truth object, measuring the method's accuracy |

#### 📌 Example of Expected Output

```
Objeto 1: tipo=Tria, validado=True
...
Acurácia: 88.89%
```

**Fixed parameters for reproducibility:** so that the output matches the automatic grading rubric, use exactly: minimum area filter of 300 pixels; `cv2.approxPolyDP` with `epsilon = 0.02 * perimeter`; solidity threshold of 0.92 and vertex count ≥ 9 (with ≥ 11 to distinguish Cruz from Estrela) for concave shapes; aspect ratio of 1.15 to distinguish Círculo from Elipse; IoU threshold of 0.5 for validation.

#### 📌 Reference Files (`.jpg` and `.txt`)

For local debugging, two reference files are provided (included in this submission; when integrating them into the chapter repository, save them in `all/cap08/dados/EP08/`):

* 📥 **Image (`00000.jpg`)**: image of geometric objects used as the exercise input. The goal is to automatically segment each object, determine its type, and compute its measurements.
* 📥 **Ground truth (`00000.txt`)**: file containing the normalized bounding boxes of the objects in the image. Each line contains the class identifier and the normalized coordinates of the upper-left and lower-right corners, used to automatically validate the segmentation.

[Figure 8.12](#fig-08-ep12) shows the input image and the same image with the bounding boxes drawn from the ground truth file.

In [ ]:
import os
import urllib.request
from morph import mm

def garantir_e_baixar(nome):
    pasta = "dados/EP12"
    caminho = os.path.join(pasta, nome)

    os.makedirs(pasta, exist_ok=True)

    if not os.path.exists(caminho):
        url = (
            "https://raw.githubusercontent.com/"
            "fzampirolli/pdi-vc/master/all/cap08/dados/EP12/"
            + nome
        )
        print(f"Downloading {nome}...")
        urllib.request.urlretrieve(url, caminho)

    return caminho

img_arq = garantir_e_baixar("00000.jpg")
txt_arq = garantir_e_baixar("00000.txt")

img = mm.read(img_arq)
img_bb = mm.showBoundBox(img, txt_arq, fmt="yolo", show=False)

mm.show(
    [img, img_bb],
    titles=[
        "Original image",
        "Answer key bounding boxes"
    ],
    cols=2,
    figsize=(10,5)
)

**Figure 8.12:** Simulator EP08_12: Image used in EP08_12. On the left, the original image. On the right, the image with the *bounding boxes* from the answer key file.


In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0812" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulator EP08_12: Segmentation Accuracy on Multiple Objects</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">🟢 hit if IoU ≥ threshold AND correct type</span>
  </div>


  <div style="padding:20px;background:white;overflow:auto">

    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Each shape has a <i>bounding box</i> ground truth (dashed rectangle, tight around the shape) and a <i>bounding box</i> detected (solid rectangle, offset/noisy). Adjust the noise, bias, and IoU threshold to see the validation change.
    </p>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#c0392b;">segmentation noise (px, max jitter per side)</label><span id="ep0812_ruido_v" style="font-family:monospace;font-weight:bold;color:#c0392b;">0</span></div>
        <input id="ep0812_ruido" style="width:100%;accent-color:#c0392b;" max="20" min="0" step="1" type="range" value="0">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">systematic bias in x (px)</label><span id="ep0812_bias_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">0</span></div>
        <input id="ep0812_bias" style="width:100%;accent-color:#2980b9;" max="20" min="-20" step="1" type="range" value="0">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#27ae60;">IoU threshold</label><span id="ep0812_thr_v" style="font-family:monospace;font-weight:bold;color:#27ae60;">0.50</span></div>
        <input id="ep0812_thr" style="width:100%;accent-color:#27ae60;" max="0.9" min="0.1" step="0.05" type="range" value="0.5">
      </div>
      <div style="display:flex;align-items:center;gap:8px;">
        <input id="ep0812_erro" type="checkbox" style="accent-color:#8e44ad;width:16px;height:16px;">
        <label style="font-size:12px;font-weight:bold;color:#8e44ad;">simulate classification error (2 objects with swapped types)</label>
      </div>
    </div>

<div id="ep0812_svg" style="width:100%;max-width:420px;margin:0 auto 16px auto;"></div>

    <table style="width:100%;border-collapse:collapse;font-size:11px;font-family:monospace;margin-bottom:12px;">
      <thead>
        <tr style="background:#f3efe6;">
          <th style="padding:4px;border:1px solid #ddd;">id</th>
          <th style="padding:4px;border:1px solid #ddd;">true type</th>
          <th style="padding:4px;border:1px solid #ddd;">detected type</th>
          <th style="padding:4px;border:1px solid #ddd;">IoU</th>
          <th style="padding:4px;border:1px solid #ddd;">≥ threshold</th>
          <th style="padding:4px;border:1px solid #ddd;">hit</th>
        </tr>
      </thead>
      <tbody id="ep0812_tbody"></tbody>
    </table>

    <div id="ep0812_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:12px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>

function svgNS(tag){
  var SVG_NS = "http" + "://www.w3.org/2000/svg";
  return document.createElementNS(SVG_NS, tag);
}

(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var ruidoEl = root.querySelector('#ep0812_ruido'), ruidovEl = root.querySelector('#ep0812_ruido_v');
    var biasEl = root.querySelector('#ep0812_bias'), biasvEl = root.querySelector('#ep0812_bias_v');
    var thrEl = root.querySelector('#ep0812_thr'), thrvEl = root.querySelector('#ep0812_thr_v');
    var erroEl = root.querySelector('#ep0812_erro');
    var svgWrap = root.querySelector('#ep0812_svg');
    var svgEl = svgNS('svg');
    svgEl.setAttribute('viewBox', '0 0 360 340');
    svgEl.setAttribute('style', 'width:100%;display:block;background:#0d0d0d;border-radius:12px;border:1px solid #333;');
    svgWrap.appendChild(svgEl);

    var tbody = root.querySelector('#ep0812_tbody');
    var dbg = root.querySelector('#ep0812_debug');

    var TIPOS = ['Circ','Tria','Quad','Cruz','Pent','Hexa','Hept','Estr','Elip'];
    var CORES = ['#a33a5b','#8fe3b0','#c98a7a','#5c4a5e','#3fbf5f','#d9c832','#e08a2b','#5c5470','#4a5a3a'];
    var FATOR = [1.0, 1.3, 0.7, 1.5, 0.9, 1.1, 0.6, 1.4, 0.8]; // sensibilidade individual ao ruído (fixa)
    var ANGS  = [30, 160, 260, 5, 200, 90, 340, 130, 240];      // direção fixa do erro por objeto (graus)
    var SCALE = [0.9, 1.15, 0.8, 1.2, 1.0, 0.95, 1.1, 1.25, 0.85]; // fator de encolhimento/expansão do bbox (fixo)

    var OBJS = [
      {cx:55,  cy:45,  r:22},
      {cx:150, cy:70,  r:24},
      {cx:250, cy:50,  r:22},
      {cx:320, cy:130, r:20},
      {cx:90,  cy:190, r:24},
      {cx:190, cy:220, r:24},
      {cx:275, cy:190, r:24},
      {cx:315, cy:270, r:22},
      {cx:150, cy:150, r:18}
    ];

    function svgShape(tipo, cx, cy, r, cor){
      var s = '';
      if(tipo==='Circ'){
        s = '<circle cx="'+cx+'" cy="'+cy+'" r="'+r+'" fill="'+cor+'"/>';
      } else if(tipo==='Elip'){
        s = '<ellipse cx="'+cx+'" cy="'+cy+'" rx="'+(r*1.1)+'" ry="'+(r*0.65)+'" fill="'+cor+'"/>';
      } else if(tipo==='Quad'){
        s = '<rect x="'+(cx-r*0.8)+'" y="'+(cy-r*0.8)+'" width="'+(r*1.6)+'" height="'+(r*1.6)+'" fill="'+cor+'"/>';
      } else if(tipo==='Cruz'){
        var w = r*0.5, l = r*1.4;
        s = '<g fill="'+cor+'">'+
            '<rect x="'+(cx-w/2)+'" y="'+(cy-l/2)+'" width="'+w+'" height="'+l+'"/>'+
            '<rect x="'+(cx-l/2)+'" y="'+(cy-w/2)+'" width="'+l+'" height="'+w+'"/></g>';
      } else {
        var sides = {Tria:3, Pent:5, Hexa:6, Hept:7}[tipo];
        var isStar = (tipo==='Estr');
        var pts = [];
        if(isStar){
          var spikes=5, outer=r, inner=r*0.45;
          for(var i=0;i<spikes*2;i++){
            var rad = (i%2===0)?outer:inner;
            var ang = Math.PI/spikes*i - Math.PI/2;
            pts.push((cx+rad*Math.cos(ang)).toFixed(1)+','+(cy+rad*Math.sin(ang)).toFixed(1));
          }
        } else {
          for(var i=0;i<sides;i++){
            var ang = 2*Math.PI/sides*i - Math.PI/2;
            pts.push((cx+r*Math.cos(ang)).toFixed(1)+','+(cy+r*Math.sin(ang)).toFixed(1));
          }
        }
        s = '<polygon points="'+pts.join(' ')+'" fill="'+cor+'"/>';
      }
      return s;
    }

    // bbox "verdadeiro": retângulo justo em torno da forma (sem folga artificial)
    function trueBBox(tipo, cx, cy, r){
      var hw = r, hh = r;
      if(tipo==='Elip'){ hw = r*1.1; hh = r*0.65; }
      else if(tipo==='Quad'){ hw = r*0.8; hh = r*0.8; }
      else if(tipo==='Cruz'){ hw = r*0.7; hh = r*0.7; }
      return [cx-hw, cy-hh, cx+hw, cy+hh];
    }

    function iou(a, b){
      var x1 = Math.max(a[0], b[0]), y1 = Math.max(a[1], b[1]);
      var x2 = Math.min(a[2], b[2]), y2 = Math.min(a[3], b[3]);
      var inter = Math.max(0, x2-x1) * Math.max(0, y2-y1);
      var areaA = (a[2]-a[0])*(a[3]-a[1]);
      var areaB = (b[2]-b[0])*(b[3]-b[1]);
      var uni = areaA + areaB - inter;
      return uni > 0 ? inter/uni : 0;
    }

    function render(){
      var ruido = parseInt(ruidoEl.value);
      var bias = parseInt(biasEl.value);
      var thr = parseFloat(thrEl.value);
      var erroAtivo = erroEl.checked;
      ruidovEl.textContent = ruido;
      biasvEl.textContent = bias;
      thrvEl.textContent = thr.toFixed(2);

      var svgContent = '';
      var rows = '';
      var acertos = 0;

      OBJS.forEach(function(o, i){
        var tipoReal = TIPOS[i];
        var cor = CORES[i];

        var gtBox = trueBBox(tipoReal, o.cx, o.cy, o.r);
        svgContent += '<g opacity="0.9">'+svgShape(tipoReal, o.cx, o.cy, o.r, cor)+'</g>';
        svgContent += '<rect x="'+gtBox[0]+'" y="'+gtBox[1]+'" width="'+(gtBox[2]-gtBox[0])+'" height="'+(gtBox[3]-gtBox[1])+'" fill="none" stroke="#aaa" stroke-dasharray="4,3" stroke-width="1.2"/>';

        // bbox detectado: escala fixa individual + jitter (ruído) + viés em x
        var scl = SCALE[i];
        var mag = ruido * FATOR[i];
        var ang = ANGS[i] * Math.PI/180;
        var jx = mag*Math.cos(ang), jy = mag*Math.sin(ang);
        var dw = (gtBox[2]-gtBox[0]) * scl, dh = (gtBox[3]-gtBox[1]) * scl;
        var dcx = o.cx + jx + bias, dcy = o.cy + jy;
        var detBox = [dcx-dw/2, dcy-dh/2, dcx+dw/2, dcy+dh/2];

        var val = iou(gtBox, detBox);
        var passaLimiar = val >= thr;

        var corDet = passaLimiar ? '#27ae60' : '#c0392b';
        svgContent += '<rect x="'+detBox[0]+'" y="'+detBox[1]+'" width="'+(detBox[2]-detBox[0])+'" height="'+(detBox[3]-detBox[1])+'" fill="none" stroke="'+corDet+'" stroke-width="1.6"/>';

        var tipoDetectado = tipoReal;
        if(erroAtivo && (i===1 || i===6)){
          tipoDetectado = TIPOS[(i+2)%TIPOS.length];
        }
        var tipoCorreto = (tipoDetectado === tipoReal);
        var acertou = passaLimiar && tipoCorreto;
        if(acertou) acertos++;

        svgContent += '<text x="'+(o.cx)+'" y="'+(gtBox[1]-6)+'" font-size="9" fill="#ccc" text-anchor="middle" font-family="monospace">'+ (i+1) +'</text>';

        rows += '<tr>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+(i+1)+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+tipoReal+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;'+(tipoCorreto?'':'color:#c0392b;font-weight:bold;')+'">'+tipoDetectado+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+val.toFixed(2)+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+(passaLimiar?'sim':'não')+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;'+(acertou?'color:#27ae60;font-weight:bold;':'color:#c0392b;font-weight:bold;')+'">'+(acertou?'✔':'✘')+'</td>'+
          '</tr>';
      });

      svgEl.innerHTML = svgContent;
      tbody.innerHTML = rows;

      var acc = (acertos/OBJS.length*100).toFixed(1);
      dbg.textContent = 'Objetos validados: '+acertos+' / '+OBJS.length+'  →  Acurácia = '+acc+'%';
    }

    ruidoEl.addEventListener('input', render);
    biasEl.addEventListener('input', render);
    thrEl.addEventListener('input', render);
    erroEl.addEventListener('change', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0812');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 8.13:** EP08_12 Simulator: Segmentation Accuracy in Multiple Objects (IoU)


<figure id="fig-08-sim-ep0812">
  <img src="imagens/fig-08-sim-ep0812.png" alt=" EP08_12 Simulator: Segmentation Accuracy in Multiple Objects (IoU) " style="max-width:80%" />
  <figcaption><strong>Figure 8.13:</strong>  EP08_12 Simulator: Segmentation Accuracy in Multiple Objects (IoU) </figcaption>
</figure>

In [ ]:
%%writefile EP08_12.py
# Python code

In [ ]:
TestSuite("EP08_12.py").run()